In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from typing import Dict, Any, Optional

import pandas as pd
import numpy as np
from typing import Dict, Any, Optional

from FinancialCalculatorBase import FinancialCalculatorBase
from FinancialDataManager import FinancialDataManager

import os

In [2]:
import pandas as pd

# Official QQQ holdings CSV
filename = "invesco_qqq_trust,_series_1-monthly_holdings.csv"

df = pd.read_csv(filename)

# Keep only relevant columns
df = df[['Ticker', '%TNA', 'Share_Par','Market_value']]

# Clean weight column (remove % sign and convert to float)
df['TNA'] = df['%TNA'].str.replace('%', '').astype(float)

# Sort by weight (largest first)
df = df.sort_values('TNA', ascending=False)

print(df.head(10))   # Top 10 stocks
print("Total stocks:", len(df))


  Ticker   %TNA       Share_Par        Market_value   TNA
0   NVDA  8.75%  188,255,668.00  $34,415,018,667.08  8.75
1   AAPL  7.44%  114,474,151.00  $29,280,198,342.78  7.44
2   MSFT  5.87%   57,578,508.00  $23,107,406,830.56  5.87
3   AMZN  4.18%   82,816,840.00  $16,463,159,623.60  4.18
4   TSLA  4.10%   38,665,898.00  $16,140,692,461.12  4.10
5   META  3.70%   22,751,605.00  $14,555,794,330.85  3.70
6    WMT  3.56%  104,523,309.00  $13,994,625,842.01  3.56
7  GOOGL  3.50%   45,071,927.00  $13,779,389,522.44  3.50
8   GOOG  3.26%   41,887,923.00  $12,818,542,196.46  3.26
9   AVGO  3.02%   36,584,062.00  $11,896,039,440.54  3.02
Total stocks: 106


In [3]:
class StockDataFetcher:
    """Fetches financial data from yfinance"""
    
    def __init__(self, ticker):
        self.ticker = ticker
        self.stock = yf.Ticker(ticker)
        
        
    def get_sector_info(self):   
        try:
            #stock = yf.Ticker(self.ticker)
            info = self.stock.info
            sector_str = info.get('sector', 'Unknown')
            print("sector str", sector_str)
            return sector_str
        
        except Exception as e:
                print(f" Error fetching sector info for {ticker}: {e}")
                return 'Unknown'
                   

    def fetch_balance_sheet(self, period='quarterly'):
        """
        Fetch balance sheet data        
        Args:
            period: 'quarterly' or 'annual'
        Returns:
            DataFrame with balance sheet data
        """
        print(f"Fetching {period} balance sheet for {self.ticker}...")
        
        if period == 'quarterly':
            data = self.stock.quarterly_balance_sheet
        else:
            data = self.stock.balance_sheet
            
        if data is not None and not data.empty:
            print(f"✓ Retrieved {len(data.columns)} periods of balance sheet data")
            return data
        else:
            print("✗ No balance sheet data available")
            return None
    
    def fetch_income_statement(self, period='quarterly'):
        """
        Fetch income statement data        
        Args:
            period: 'quarterly' or 'annual'
        Returns:
            DataFrame with income statement data
        """
        print(f"Fetching {period} income statement for {self.ticker}...")
        
        if period == 'quarterly':
            data = self.stock.quarterly_income_stmt
        else:
            data = self.stock.income_stmt
            
        if data is not None and not data.empty:
            print(f"✓ Retrieved {len(data.columns)} periods of income statement data")
            return data
        else:
            print("✗ No income statement data available")
            return None
    
    def fetch_cash_flow(self, period='quarterly'):
        """
        Fetch cash flow statement data      
        Args:
            period: 'quarterly' or 'annual'
        Returns:
            DataFrame with cash flow data
        """
        print(f"Fetching {period} cash flow for {self.ticker}...")        
        if period == 'quarterly':
            data = self.stock.quarterly_cashflow
        else:
            data = self.stock.cashflow            
        if data is not None and not data.empty:
            print(f"✓ Retrieved {len(data.columns)} periods of cash flow data")
            return data
        else:
            print("✗ No cash flow data available")
            return None
    
    def fetch_stock_prices(self, period='2y', interval='1d'):
        """
        Fetch historical stock prices
        Args:
            period: Time period (1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max)
            interval: Data interval (1m, 2m, 5m, 15m, 30m, 60m, 90m, 1h, 1d, 5d, 1wk, 1mo, 3mo)
        Returns:
            DataFrame with stock price data
        """
        print(f"Fetching stock prices for {self.ticker} ({period})...")        
        data = self.stock.history(period=period, interval=interval)
        if not data.empty:
            print(f" Retrieved {len(data)} periods of price data")
            return data
        else:
            print(" No stock price data available")
            return None
    
    def fetch_all_data(self, period='quarterly'):
        """
        Fetch all financial statements at once  
        Returns:
            Dictionary containing all financial data
        """
        print(f"\n{'='*60}")
        print(f"Fetching all financial data for {self.ticker}")
        print(f"{'='*60}\n")
        
        return {
            'balance_sheet': self.fetch_balance_sheet(period),
            'income_statement': self.fetch_income_statement(period),
            'cash_flow': self.fetch_cash_flow(period),
            'stock_prices': self.fetch_stock_prices()
        }

In [4]:
"""
Robust Liquidity Ratio Calculator
Handles zero/missing Current Liabilities gracefully
"""

import pandas as pd
import numpy as np
from typing import Dict, Any, Optional
import warnings

class RobustLiquidityCalculator(FinancialCalculatorBase):
    """
    Calculate liquidity ratios with comprehensive error handling
    """
    
    def __init__(self, ticker: str, balance_sheet: pd.DataFrame):
        super().__init__(ticker, balance_sheet, min_threshold=1e6, max_ratio=100.0)
        self.balance_sheet = balance_sheet  # Keep for compatibility

    def calculate_current_ratio(self) -> pd.Series:
        """
        Current Ratio = Current Assets / Current Liabilities
        
        Handling:
        - If CL = 0: Cap at MAX_RATIO (indicates very strong liquidity)
        - If both zero: NaN (no data)
        - Otherwise: Normal calculation with cap
        """
        print("\n  Calculating Current Ratio...")
        
        current_assets = self.safe_get('Current Assets')
        current_liabilities = self.safe_get('Current Liabilities')
        
      # Check data availability
        has_data, missing = self.check_missing_data('Current Assets', 'Current Liabilities')
        if not has_data:
            print(f"  Missing data: {missing}")
            return pd.Series(np.nan, index=self.data.columns)
        
        if current_liabilities.isna().all():
            print(" Current Liabilities missing for all periods")
            return pd.Series(np.nan, index=current_assets.index)
        
        # Calculate with safe division
        ratio = self.safe_divide(current_assets, current_liabilities)
        
        # Check for issues
        zero_liability_count = (current_liabilities < self.MIN_THRESHOLD).sum()
        if zero_liability_count > 0:
            print(f" {zero_liability_count} periods with zero/near-zero Current Liabilities")
            print(f" Capped ratios at {self.MAX_RATIO}")
        
        # Post-processing: interpolate NaN values
        ratio = self._interpolate_missing(ratio, 'Current Ratio')
        
        print(f"  Current Ratio calculated")
        print(f"  Range: {ratio.min():.2f} to {ratio.max():.2f}")
        return ratio
    
    def calculate_quick_ratio(self) -> pd.Series:
        """
        Quick Ratio = (Current Assets - Inventory) / Current Liabilities
        
        Handling:
        - Missing inventory → assume zero inventory (service company)
        - CL = 0 → cap at MAX_RATIO
        """
        print("\n  Calculating Quick Ratio...")
        
        current_assets = self.safe_get('Current Assets')
        inventory = self.safe_get('Inventory', default=0)  # Default to 0 for service companies
        current_liabilities = self.safe_get('Current Liabilities')
        
        # Check if we have essential data
        if current_assets.isna().all() or current_liabilities.isna().all():
            print(" Missing essential data")
            return pd.Series(np.nan, index=current_assets.index)
        
        # Calculate quick assets
        quick_assets = current_assets - inventory
        
        # Calculate with safe division
        ratio = self.safe_divide(quick_assets, current_liabilities)
        
        # Check for issues
        zero_liability_count = (current_liabilities < self.MIN_THRESHOLD).sum()
        if zero_liability_count > 0:
            print(f"  {zero_liability_count} periods with zero/near-zero Current Liabilities")
            print(f" → Capped ratios at {self.MAX_RATIO}")
        
        # Post-processing
        ratio = self._interpolate_missing(ratio, 'Quick Ratio')
        print(f"  Quick Ratio calculated")
        print(f" Range: {ratio.min():.2f} to {ratio.max():.2f}")
        
        return ratio
    
    def calculate_cash_ratio(self) -> pd.Series:
        """
        Cash Ratio = Cash / Current Liabilities
        Handling: - CL = 0 → cap at MAX_RATIO
        """
        print("\n  Calculating Cash Ratio...")
        
        cash = self.safe_get('Cash And Cash Equivalents')
        current_liabilities = self.safe_get('Current Liabilities')
        
        # Check if we have data
        if cash.isna().all() or current_liabilities.isna().all():
            print("  Missing essential data")
            return pd.Series(np.nan, index=cash.index)
        
        # Calculate with safe division
        ratio = self.safe_divide(cash, current_liabilities)
        
        # Check for issues
        zero_liability_count = (current_liabilities < self.MIN_THRESHOLD).sum()
        if zero_liability_count > 0:
            print(f" {zero_liability_count} periods with zero/near-zero Current Liabilities")
            print(f"  Capped ratios at {self.MAX_RATIO}")
        
        # Post-processing
        ratio = self._interpolate_missing(ratio, 'Cash Ratio')
        
        print(f" Cash Ratio calculated")
        print(f"  Range: {ratio.min():.2f} to {ratio.max():.2f}")
        
        return ratio
    
    def calculate_working_capital(self) -> pd.Series:
        """
        Working Capital = Current Assets - Current Liabilities
        This is SAFE even when CL = 0 (just returns Current Assets)
        """
        print("\n  Calculating Working Capital...")
        
        current_assets = self.safe_get('Current Assets')
        current_liabilities = self.safe_get('Current Liabilities')
        
        working_capital = current_assets - current_liabilities
        
        print(f" Working Capital calculated")
        return working_capital

    
    def _interpolate_missing(self, series: pd.Series, ratio_name: str) -> pd.Series:
        """
        Interpolate missing values in time series    
        Strategy:
        1. Linear interpolation for gaps
        2. Forward/backward fill for edges
        3. If still NaN, use median
        """
        original_nan_count = series.isna().sum()
        
        if original_nan_count == 0:
            return series
        
        # Sort by date (assuming index is dates)
        series = series.sort_index()
        
        # Linear interpolation
        series = series.interpolate(method='linear', limit_direction='both')
        
        # Forward fill
        series = series.fillna(method='ffill')
        
        # Backward fill
        series = series.fillna(method='bfill')
        
        # If still NaN, use median
        if series.isna().any():
            median = series.median()
            if not pd.isna(median):
                series = series.fillna(median)
            else:
                # Last resort: use a reasonable default
                default_values = {
                    'Current Ratio': 1.5,
                    'Quick Ratio': 1.2,
                    'Cash Ratio': 0.5
                }
                series = series.fillna(default_values.get(ratio_name, 1.0))
        
        filled_count = original_nan_count - series.isna().sum()
        if filled_count > 0:
            print(f" Filled {filled_count} missing values via interpolation")
        
        return series
    
    def calculate_all_liquidity_ratios(self) -> Dict[str, pd.Series]:
        """
        Calculate all liquidity ratios with comprehensive error handling
        """
        print(f"\n{'='*60}")
        print(f"Calculating Liquidity Ratios for {self.ticker}")
        print(f"{'='*60}")
        
        ratios = {}
        
        # Current Ratio
        try:
            ratios['Current_Ratio'] = self.calculate_current_ratio()
        except Exception as e:
            print(f" Current Ratio failed: {e}")
            ratios['Current_Ratio'] = pd.Series(np.nan, index=self.balance_sheet.columns)
        
        # Quick Ratio
        try:
            ratios['Quick_Ratio'] = self.calculate_quick_ratio()
        except Exception as e:
            print(f" Quick Ratio failed: {e}")
            ratios['Quick_Ratio'] = pd.Series(np.nan, index=self.balance_sheet.columns)
        
        # Cash Ratio
        try:
            ratios['Cash_Ratio'] = self.calculate_cash_ratio()
        except Exception as e:
            print(f" Cash Ratio failed: {e}")
            ratios['Cash_Ratio'] = pd.Series(np.nan, index=self.balance_sheet.columns)
        
        # Working Capital
        try:
            ratios['Working_Capital'] = self.calculate_working_capital()
        except Exception as e:
            print(f"  Working Capital failed: {e}")
            ratios['Working_Capital'] = pd.Series(np.nan, index=self.balance_sheet.columns)
        
        # Validate and enforce constraints
        ratios = self._validate_ratios(ratios)
        
        print(f"\n{'='*60}")
        print(f"All liquidity ratios calculated")
        print(f"{'='*60}\n")
        
        return ratios
    
    def _validate_ratios(self, ratios: Dict[str, pd.Series]) -> Dict[str, pd.Series]:
        """
        Validate that ratios follow expected constraints     
        Constraints:
        1. Cash Ratio ≤ Quick Ratio ≤ Current Ratio
        2. All ratios ≥ 0
        3. Ratios capped at MAX_RATIO
        """
        print("\n  Validating ratio constraints...")
        
        violations = 0
        
        for idx in ratios['Current_Ratio'].index:
            current = ratios['Current_Ratio'][idx]
            quick = ratios['Quick_Ratio'][idx]
            cash = ratios['Cash_Ratio'][idx]
            
            # Skip if any are NaN
            if pd.isna(current) or pd.isna(quick) or pd.isna(cash):
                continue
            
            # Check constraint: Cash ≤ Quick ≤ Current
            if quick > current:
                # Quick should never exceed Current
                ratios['Quick_Ratio'][idx] = current
                violations += 1
            
            if cash > quick:
                # Cash should never exceed Quick
                ratios['Cash_Ratio'][idx] = quick
                violations += 1
            
            # Ensure non-negative
            ratios['Current_Ratio'][idx] = max(0, current)
            ratios['Quick_Ratio'][idx] = max(0, quick)
            ratios['Cash_Ratio'][idx] = max(0, cash)
        
        if violations > 0:
            print(f" Fixed {violations} constraint violations")
        else:
            print(f" All constraints satisfied")
        
        return ratios
    
    def get_data_quality_report(self) -> Dict[str, Any]:
        """
        Generate data quality report
        """
        report = {
            'ticker': self.ticker,
            'total_periods': len(self.balance_sheet.columns),
            'issues': []
        }
        
        # Check Current Liabilities
        cl = self.safe_get('Current Liabilities')
        zero_cl = (cl < self.MIN_THRESHOLD).sum()
        missing_cl = cl.isna().sum()
        
        if zero_cl > 0:
            report['issues'].append({
                'issue': 'Zero Current Liabilities',
                'count': zero_cl,
                'severity': 'HIGH',
                'action': f'Ratios capped at {self.MAX_RATIO}'
            })
        
        if missing_cl > 0:
            report['issues'].append({
                'issue': 'Missing Current Liabilities',
                'count': missing_cl,
                'severity': 'HIGH',
                'action': 'Values interpolated'
            })
        
        # Check Current Assets
        ca = self.safe_get('Current Assets')
        missing_ca = ca.isna().sum()
        
        if missing_ca > 0:
            report['issues'].append({
                'issue': 'Missing Current Assets',
                'count': missing_ca,
                'severity': 'HIGH',
                'action': 'Values interpolated'
            })
        
        return report


# ============================================================================
# INTEGRATION WITH YOUR EXISTING CODE
# ============================================================================

def integrate_with_existing_calculator(self):
    """
    How to integrate into your FinancialRatioCalculator class
    """
    
    # Inside your calculate_ratios method:
    
    print("\n" + "="*60)
    print("LIQUIDITY RATIOS")
    print("="*60)
    
    # Use robust calculator
    liquidity_calc = RobustLiquidityCalculator(self.ticker, self.balance_sheet)
    
    # Get all liquidity ratios
    liquidity_ratios = liquidity_calc.calculate_all_liquidity_ratios()
    quality_report = liquidity_calc.get_data_quality_report()

    '''
    # Add to your ratios dictionary
    ratios = {}
    ratios['Current_Ratio'] = liquidity_ratios['Current_Ratio']
    ratios['Quick_Ratio'] = liquidity_ratios['Quick_Ratio']
    ratios['Cash_Ratio'] = liquidity_ratios['Cash_Ratio']
    ratios['Working_Capital'] = liquidity_ratios['Working_Capital']
    
    # Get data quality report
    '''
    
    # Log issues
    if quality_report['issues']:
        print("\n Data Quality Issues:")
        for issue in quality_report['issues']:
            print(f"  • {issue['issue']}: {issue['count']} periods")
            print(f"    Action: {issue['action']}")

    #result is a pandas series 
    return liquidity_ratios



In [5]:

class RobustLeverageCalculator(FinancialCalculatorBase):
    """
    Calculate leverage ratios with comprehensive error handling
    Handles zero equity, zero assets, missing data
    """
    
    def __init__(self, ticker: str, balance_sheet: pd.DataFrame):
        super().__init__(ticker, balance_sheet, min_threshold=1e6, max_ratio=100.0)
        self.balance_sheet = balance_sheet
    
    def calculate_debt_to_equity(self) -> pd.Series:
        """
        Debt-to-Equity = Total Liabilities / Stockholders Equity
        
        Handling:
        - If Equity = 0: NaN (company has no equity - bankrupt or special entity)
        - If Equity < 0: Allow negative ratio (shows insolvency)
        - Otherwise: Normal calculation with cap
        """
        print("\n  Calculating Debt-to-Equity Ratio...")
        
        total_liabilities = self.safe_get('Total Liabilities Net Minority Interest')
        equity = self.safe_get('Stockholders Equity')
        
        # Check if we have data
        if total_liabilities.isna().all() or equity.isna().all():
            print("   Missing essential data")
            return pd.Series(np.nan, index=equity.index)
        
        # Calculate with safe division
        ratio = self.safe_divide(total_liabilities, equity)
        
        # Check for issues
        zero_equity_count = (equity.abs() < self.MIN_THRESHOLD).sum()
        negative_equity_count = (equity < 0).sum()
        
        if zero_equity_count > 0:
            print(f"   {zero_equity_count} periods with zero/near-zero Equity")
            print(f"       → Ratios marked as NaN (will be interpolated)")
        
        if negative_equity_count > 0:
            print(f"   {negative_equity_count} periods with NEGATIVE Equity")
            print(f"       → Company may be insolvent in these periods")
        
        # Post-processing: interpolate NaN values
        ratio = self._interpolate_missing(ratio, 'Debt_to_Equity')
        
        print(f"    Debt-to-Equity Ratio calculated")
        if not ratio.isna().all():
            print(f"       Range: {ratio.min():.2f} to {ratio.max():.2f}")
        
        return ratio
    
    def calculate_debt_to_assets(self) -> pd.Series:
        """
        Debt-to-Assets = Total Liabilities / Total Assets
        
        Handling:
        - If Assets = 0: NaN (very unusual)
        - Result should be between 0 and 1 (0% to 100%)
        - If > 1: Company owes more than it owns (insolvency)
        """
        print("\n  Calculating Debt-to-Assets Ratio...")
        
        total_liabilities = self.safe_get('Total Liabilities Net Minority Interest')
        total_assets = self.safe_get('Total Assets')
        
        # Check if we have data
        if total_liabilities.isna().all() or total_assets.isna().all():
            print("  Missing essential data")
            return pd.Series(np.nan, index=total_assets.index)
        
        # Calculate with safe division
        ratio = self.safe_divide(total_liabilities, total_assets)
        
        # Check for issues
        zero_assets_count = (total_assets.abs() < self.MIN_THRESHOLD).sum()
        insolvency_count = (ratio > 1.0).sum()
        
        if zero_assets_count > 0:
            print(f" {zero_assets_count} periods with zero/near-zero Assets")
            print(f"       → Very unusual, check data quality")
        
        if insolvency_count > 0:
            print(f"  {insolvency_count} periods with ratio > 1.0")
            print(f"  Liabilities exceed Assets (insolvency signal)")
        
        # Post-processing
        ratio = self._interpolate_missing(ratio, 'Debt_to_Assets')
        
        # Validate range (should be 0 to ~1, allow up to 2 for insolvent companies)
        ratio = ratio.clip(lower=0, upper=2.0)
        
        print(f"  Debt-to-Assets Ratio calculated")
        if not ratio.isna().all():
            print(f"       Range: {ratio.min():.2f} to {ratio.max():.2f}")
        
        return ratio
    
    def _interpolate_missing(self, series: pd.Series, ratio_name: str) -> pd.Series:
        """
        Interpolate missing leverage ratio values
        
        Strategy:
        1. Linear interpolation for gaps
        2. Forward/backward fill for edges
        3. If still NaN, use reasonable defaults
        """
        original_nan_count = series.isna().sum()
        
        if original_nan_count == 0:
            return series
        
        # Sort by date
        series = series.sort_index()
        
        # Linear interpolation
        series = series.interpolate(method='linear', limit_direction='both')
        
        # Forward/backward fill
        series = series.fillna(method='ffill').fillna(method='bfill')
        
        # If still NaN, use defaults
        if series.isna().any():
            default_values = {
                'Debt_to_Equity': 0.5,     # Moderate leverage
                'Debt_to_Assets': 0.4,     # 40% debt financing
            }
            default = default_values.get(ratio_name, 0.5)
            series = series.fillna(default)
        
        filled_count = original_nan_count - series.isna().sum()
        if filled_count > 0:
            print(f"       → Filled {filled_count} missing values")
        
        return series
    
    def calculate_all_leverage_ratios(self) -> Dict[str, pd.Series]:
        """
        Calculate all leverage ratios with comprehensive error handling
        """
        print(f"\n{'='*60}")
        print(f"Calculating Leverage Ratios for {self.ticker}")
        print(f"{'='*60}")
        
        ratios = {}
        
        # Debt-to-Equity
        try:
            ratios['Debt_to_Equity'] = self.calculate_debt_to_equity()
        except Exception as e:
            print(f"    Debt-to-Equity failed: {e}")
            ratios['Debt_to_Equity'] = pd.Series(np.nan, index=self.balance_sheet.columns)
        
        # Debt-to-Assets
        try:
            ratios['Debt_to_Assets'] = self.calculate_debt_to_assets()
        except Exception as e:
            print(f"Debt-to-Assets failed: {e}")
            ratios['Debt_to_Assets'] = pd.Series(np.nan, index=self.balance_sheet.columns)
        
        # Calculate Equity Ratio (derived from Debt-to-Assets)
        # Equity Ratio = 1 - Debt-to-Assets
        ratios['Equity_Ratio'] = 1 - ratios['Debt_to_Assets']
        print("\n  ✓ Equity Ratio calculated (derived)")
        print(f"     Formula: 1 - Debt_to_Assets")
        
        # Validate relationships
        ratios = self._validate_leverage_ratios(ratios)
        self.get_data_quality_report()
        
        print(f"\n{'='*60}")
        print(f"✓ All leverage ratios calculated")
        print(f"{'='*60}\n")
        
        return ratios
    
    def _validate_leverage_ratios(self, ratios: Dict[str, pd.Series]) -> Dict[str, pd.Series]:
        """
        Validate leverage ratio relationships
        
        Constraints:
        1. Debt_to_Assets + Equity_Ratio = 1.0 (always)
        2. All ratios >= 0 (negative debt doesn't make sense)
        3. Debt_to_Assets <= 2.0 (cap for insolvent companies)
        """
        print("\n  Validating leverage ratio constraints...")
        
        violations = 0
        
        for idx in ratios['Debt_to_Assets'].index:
            debt_to_assets = ratios['Debt_to_Assets'][idx]
            equity_ratio = ratios['Equity_Ratio'][idx]
            
            # Skip if any are NaN
            if pd.isna(debt_to_assets) or pd.isna(equity_ratio):
                continue
            
            # Constraint 1: Sum should equal 1.0
            sum_ratio = debt_to_assets + equity_ratio
            if abs(sum_ratio - 1.0) > 0.01:  # Allow small floating point errors
                # Adjust to ensure they sum to 1
                total = debt_to_assets + equity_ratio
                ratios['Debt_to_Assets'][idx] = debt_to_assets / total
                ratios['Equity_Ratio'][idx] = equity_ratio / total
                violations += 1
            
            # Constraint 2: Non-negative
            ratios['Debt_to_Assets'][idx] = max(0, debt_to_assets)
            ratios['Equity_Ratio'][idx] = max(0, equity_ratio)
            
            # Constraint 3: Debt-to-Assets reasonable range
            if debt_to_assets > 2.0:
                ratios['Debt_to_Assets'][idx] = 2.0
                ratios['Equity_Ratio'][idx] = -1.0  # Negative equity (insolvent)
                violations += 1
        
        if violations > 0:
            print(f"    → Fixed {violations} constraint violations")
        else:
            print(f"    ✓ All constraints satisfied")
        
        # Verify relationship one more time
        verification_passed = True
        for idx in ratios['Debt_to_Assets'].index:
            d2a = ratios['Debt_to_Assets'][idx]
            eq = ratios['Equity_Ratio'][idx]
            
            if pd.notna(d2a) and pd.notna(eq):
                if abs((d2a + eq) - 1.0) > 0.01:
                    verification_passed = False
                    break
        
        if verification_passed:
            print(f"   Debt_to_Assets + Equity_Ratio = 1.0 verified")
        else:
            print(f"    Some periods don't satisfy relationship")
        
        return ratios
    
    def get_data_quality_report(self) -> Dict[str, Any]:
        """
        Generate data quality report for leverage ratios
        """
        report = {
            'ticker': self.ticker,
            'total_periods': len(self.balance_sheet.columns),
            'issues': []
        }
        
        # Check Total Liabilities
        liabilities = self.safe_get('Total Liabilities Net Minority Interest')
        missing_liabilities = liabilities.isna().sum()
        
        if missing_liabilities > 0:
            report['issues'].append({
                'issue': 'Missing Total Liabilities',
                'count': missing_liabilities,
                'severity': 'HIGH',
                'action': 'Values interpolated'
            })
        
        # Check Stockholders Equity
        equity = self.safe_get('Stockholders Equity')
        zero_equity = (equity.abs() < self.MIN_THRESHOLD).sum()
        negative_equity = (equity < 0).sum()
        missing_equity = equity.isna().sum()
        
        if zero_equity > 0:
            report['issues'].append({
                'issue': 'Zero Stockholders Equity',
                'count': zero_equity,
                'severity': 'CRITICAL',
                'action': 'Ratios marked as NaN, interpolated'
            })
        
        if negative_equity > 0:
            report['issues'].append({
                'issue': 'Negative Stockholders Equity',
                'count': negative_equity,
                'severity': 'HIGH',
                'action': 'Insolvency signal - negative ratios preserved'
            })
        
        if missing_equity > 0:
            report['issues'].append({
                'issue': 'Missing Stockholders Equity',
                'count': missing_equity,
                'severity': 'HIGH',
                'action': 'Values interpolated'
            })
        
        # Check Total Assets
        assets = self.safe_get('Total Assets')
        zero_assets = (assets.abs() < self.MIN_THRESHOLD).sum()
        missing_assets = assets.isna().sum()
        
        if zero_assets > 0:
            report['issues'].append({
                'issue': 'Zero Total Assets',
                'count': zero_assets,
                'severity': 'CRITICAL',
                'action': 'Very unusual - check data source'
            })
        
        if missing_assets > 0:
            report['issues'].append({
                'issue': 'Missing Total Assets',
                'count': missing_assets,
                'severity': 'HIGH',
                'action': 'Values interpolated'
            })
        
        return report


# ============================================================================
# INTEGRATION WITH YOUR EXISTING CALCULATOR
# ============================================================================

def integrate_leverage_calculator(self):
    """
    How to integrate into your FinancialRatioCalculator class
    """
    
    print("\n" + "="*60)
    print("LEVERAGE RATIOS")
    print("="*60)
    
    # Use robust calculator
    leverage_calc = RobustLeverageCalculator(self.ticker, self.balance_sheet)
    
    # Get all leverage ratios
    leverage_ratios = leverage_calc.calculate_all_leverage_ratios()
    
    # Add to your ratios dictionary
    ratios = {}
    ratios['Debt_to_Equity'] = leverage_ratios['Debt_to_Equity']
    ratios['Debt_to_Assets'] = leverage_ratios['Debt_to_Assets']
    ratios['Equity_Ratio'] = leverage_ratios['Equity_Ratio']
    
    # Get data quality report
    quality_report = leverage_calc.get_data_quality_report()
    
    # Log issues
    if quality_report['issues']:
        print("\n Data Quality Issues:")
        for issue in quality_report['issues']:
            print(f"  • {issue['issue']}: {issue['count']} periods")
            print(f"    Severity: {issue['severity']}")
            print(f"    Action: {issue['action']}")
    
    return ratios




In [6]:
"""
Robust Profitability and Efficiency Ratio Calculators
Using FinancialCalculatorBase utilities
"""

import pandas as pd
import numpy as np



# ============================================================================
# PROFITABILITY RATIOS CALCULATOR
# ============================================================================

class RobustProfitabilityCalculator(FinancialCalculatorBase):
    """
    Calculate profitability ratios with comprehensive error handling
    
    Ratios calculated:
    - ROE (Return on Equity)
    - ROA (Return on Assets)
    - Net Profit Margin
    - Gross Profit Margin
    - Operating Margin
    - EBITDA Margin
    """
    
    def __init__(self, ticker: str, income_statement: pd.DataFrame, 
                 balance_sheet: pd.DataFrame):
        """
        Args:
            ticker: Stock ticker
            income_statement: Income statement data
            balance_sheet: Balance sheet data (needed for ROE, ROA)
        """
        super().__init__(ticker, income_statement, min_threshold=1e6, max_ratio=1000.0)
        self.income_statement = income_statement
    
    def calculate_roe(self) -> pd.Series:
        """
        ROE (Return on Equity) = (Net Income / Stockholders Equity) × 100
        
        Measures: How efficiently company uses shareholders' equity
        
        Handling:
        - If Equity = 0 or negative: NaN (company insolvent)
        - Otherwise: Normal calculation
        """
        print("\n  Calculating ROE (Return on Equity)...")
        if self.data is None or self.data.empty:
            return pd.Series(dtype=float)
            
        # Net income from income statement
        net_income = self.safe_get('Net Income')
        
        # Equity from balance sheet
        if self.balance_sheet is not None and 'Stockholders Equity' in self.balance_sheet.index:
            equity = self.balance_sheet.loc['Stockholders Equity']
            
            # Align indices (income statement and balance sheet may have different dates)
            equity = equity.reindex(net_income.index)
        else:
            print("  Stockholders Equity not available in balance sheet")
            return pd.Series(np.nan, index=net_income.index)
        
        # Calculate with safe division
        roe = self.safe_divide(net_income, equity, strategy='nan', max_value=200)
        roe = roe * 100  # Convert to percentage
        
        # Check for negative equity
        negative_equity_count = (equity < 0).sum()
        if negative_equity_count > 0:
            print(f"  {negative_equity_count} periods with NEGATIVE equity")
            print(f"       → Company may be insolvent")
        
        # Interpolate missing
        roe = self.interpolate_missing(roe, 'ROE', default=15.0)
        
        print(f"    ✓ ROE: {roe.min():.1f}% to {roe.max():.1f}%")
        return roe
    
    def calculate_roa(self) -> pd.Series:
        """
        ROA (Return on Assets) = (Net Income / Total Assets) × 100
        
        Measures: How efficiently company uses assets to generate profit
        
        Handling:
        - If Assets = 0: NaN (very unusual)
        - Otherwise: Normal calculation
        """
        print("\n  Calculating ROA (Return on Assets)...")
        
        net_income = self.safe_get('Net Income')
        
        # Assets from balance sheet
        if self.balance_sheet is not None and 'Total Assets' in self.balance_sheet.index:
            assets = self.balance_sheet.loc['Total Assets']
            assets = assets.reindex(net_income.index)
        else:
            print(" Total Assets not available in balance sheet")
            return pd.Series(np.nan, index=net_income.index)
        
        # Calculate with safe division
        roa = self.safe_divide(net_income, assets, strategy='nan', max_value=100)
        roa = roa * 100
        
        # Interpolate missing
        roa = self.interpolate_missing(roa, 'ROA', default=8.0)
        
        print(f"    ✓ ROA: {roa.min():.1f}% to {roa.max():.1f}%")
        return roa
    
    def calculate_net_profit_margin(self) -> pd.Series:
        """
        Net Profit Margin = (Net Income / Total Revenue) × 100
        
        Measures: Percentage of revenue that becomes profit
        """
        print("\n  Calculating Net Profit Margin...")
        
        net_income = self.safe_get('Net Income')
        revenue = self.safe_get('Total Revenue')
        
        # Use 'nan' strategy for zero revenue
        margin = self.safe_divide(net_income, revenue, strategy='nan', max_value=1000)
        margin = margin * 100
        
        # Check for zero revenue
        zero_revenue_count = (revenue.abs() < self.MIN_THRESHOLD).sum()
        if zero_revenue_count > 0:
            print(f"   {zero_revenue_count} periods with zero/near-zero revenue")
        
        margin = self.interpolate_missing(margin, 'Net_Profit_Margin', default=15.0)
        
        print(f"    ✓ Net Profit Margin: {margin.min():.1f}% to {margin.max():.1f}%")
        return margin
    
    def calculate_gross_profit_margin(self) -> pd.Series:
        """
        Gross Profit Margin = (Gross Profit / Total Revenue) × 100
        
        Measures: Profitability after COGS, before operating expenses
        """
        print("\n  Calculating Gross Profit Margin...")
        
        gross_profit = self.safe_get('Gross Profit')
        revenue = self.safe_get('Total Revenue')
        
        margin = self.safe_divide(gross_profit, revenue, strategy='nan', max_value=1000)
        margin = margin * 100
        margin = self.interpolate_missing(margin, 'Gross_Profit_Margin', default=40.0)
        
        print(f"    ✓ Gross Profit Margin: {margin.min():.1f}% to {margin.max():.1f}%")
        return margin
    
    def calculate_operating_margin(self) -> pd.Series:
        """
        Operating Margin = (Operating Income / Total Revenue) × 100
        
        Measures: Profitability from core operations
        """
        print("\n  Calculating Operating Margin...")
        
        operating_income = self.safe_get('Operating Income')
        revenue = self.safe_get('Total Revenue')
        
        margin = self.safe_divide(operating_income, revenue, strategy='nan', max_value=1000)
        margin = margin * 100
        margin = self.interpolate_missing(margin, 'Operating_Margin', default=20.0)
        
        print(f" Operating Margin: {margin.min():.1f}% to {margin.max():.1f}%")
        return margin
    
    def calculate_ebitda_margin(self) -> pd.Series:
        """
        EBITDA Margin = (EBITDA / Total Revenue) × 100
        
        Measures: Operating performance before non-cash charges
        
        Note: If EBITDA not available, calculate as:
        EBITDA ≈ Operating Income + Depreciation + Amortization
        """
        print("\n  Calculating EBITDA Margin...")
        
        # Try to get EBITDA directly
        ebitda = self.safe_get('EBITDA')
        revenue = self.safe_get('Total Revenue')
        
        # If EBITDA not available, try to calculate it
        if ebitda.isna().all():
            print("    → EBITDA not available, approximating from Operating Income")
            operating_income = self.safe_get('Operating Income')
            depreciation = self.safe_get('Depreciation And Amortization', default=0)
            ebitda = operating_income + depreciation
        
        margin = self.safe_divide(ebitda, revenue, strategy='nan', max_value=1000)
        margin = margin * 100
        margin = self.interpolate_missing(margin, 'EBITDA_Margin', default=25.0)
        
        print(f"    ✓ EBITDA Margin: {margin.min():.1f}% to {margin.max():.1f}%")
        return margin
    
    def calculate_all_ratios(self) -> Dict[str, pd.Series]:
        """Calculate all profitability ratios"""
        print(f"\n{'='*60}")
        print(f"Profitability Ratios for {self.ticker}")
        print(f"{'='*60}")
        
        ratios = {
            'ROE': self.calculate_roe(),
            'ROA': self.calculate_roa(),
            'Net_Profit_Margin': self.calculate_net_profit_margin(),
            'Gross_Profit_Margin': self.calculate_gross_profit_margin(),
            'Operating_Margin': self.calculate_operating_margin(),
            'EBITDA_Margin': self.calculate_ebitda_margin()
        }
        
        # Validate margin hierarchy and ROE/ROA relationship
        self._validate_profitability_constraints(ratios)
        
        print(f"\n{'='*60}")
        print(f"✓ All profitability ratios calculated")
        print(f"{'='*60}\n")
        
        return ratios
    
    def _validate_profitability_constraints(self, ratios: Dict[str, pd.Series]):
        """
        Validate profitability ratio relationships
        
        Constraints:
        1. Margin hierarchy: Net <= Operating <= Gross (for positive values)
        2. ROA <= ROE (typically, due to leverage effect)
        3. All margins should be reasonable (-100% to 100% typically)
        """
        print("\n  Validating profitability constraints...")
        
        violations = 0
        
        # Constraint 1: Margin hierarchy (only for positive margins)
        for idx in ratios['Net_Profit_Margin'].index:
            gross = ratios['Gross_Profit_Margin'][idx]
            operating = ratios['Operating_Margin'][idx]
            net = ratios['Net_Profit_Margin'][idx]
            
            if pd.notna(gross) and pd.notna(operating) and pd.notna(net):
                if gross > 0 and operating > 0 and net > 0:
                    # Operating shouldn't exceed Gross
                    if operating > gross:
                        ratios['Operating_Margin'][idx] = gross
                        violations += 1
                    # Net shouldn't exceed Operating
                    if net > operating:
                        ratios['Net_Profit_Margin'][idx] = operating
                        violations += 1
        
        # Constraint 2: ROA typically <= ROE (due to financial leverage)
        # Note: This can be violated if company has negative equity
        roa_roe_violations = 0
        for idx in ratios['ROA'].index:
            roa = ratios['ROA'][idx]
            roe = ratios['ROE'][idx]
            
            if pd.notna(roa) and pd.notna(roe):
                if roa > 0 and roe > 0 and roa > roe * 1.5:
                    # ROA significantly higher than ROE is unusual
                    # (Could indicate data issue or negative leverage)
                    roa_roe_violations += 1
        
        if violations > 0:
            print(f"    → Fixed {violations} margin hierarchy violations")
        
        if roa_roe_violations > 0:
            print(f" {roa_roe_violations} periods where ROA > ROE (check for negative leverage)")
        
        if violations == 0 and roa_roe_violations == 0:
            print(f"    ✓ All constraints satisfied")
    
    def get_data_quality_report(self) -> Dict[str, Any]:
        """Generate data quality report for profitability ratios"""
        report = {
            'ticker': self.ticker,
            'total_periods': len(self.data.columns),
            'issues': []
        }
        
        # Check revenue
        revenue = self.safe_get('Total Revenue')
        zero_revenue = (revenue.abs() < self.MIN_THRESHOLD).sum()
        missing_revenue = revenue.isna().sum()
        
        if zero_revenue > 0:
            report['issues'].append({
                'issue': 'Zero Revenue',
                'count': zero_revenue,
                'severity': 'CRITICAL',
                'action': 'Margins marked as NaN, interpolated'
            })
        
        if missing_revenue > 0:
            report['issues'].append({
                'issue': 'Missing Revenue',
                'count': missing_revenue,
                'severity': 'HIGH',
                'action': 'Values interpolated'
            })
        
        # Check net income (profitability)
        net_income = self.safe_get('Net Income')
        loss_periods = (net_income < 0).sum()
        
        if loss_periods > 0:
            report['issues'].append({
                'issue': 'Net Loss Periods',
                'count': loss_periods,
                'severity': 'MEDIUM',
                'action': 'Negative margins preserved'
            })
        
        # Check equity (for ROE)
        if self.balance_sheet is not None and 'Stockholders Equity' in self.balance_sheet.index:
            equity = self.balance_sheet.loc['Stockholders Equity']
            negative_equity = (equity < 0).sum()
            
            if negative_equity > 0:
                report['issues'].append({
                    'issue': 'Negative Equity (affects ROE)',
                    'count': negative_equity,
                    'severity': 'HIGH',
                    'action': 'ROE marked as NaN for these periods'
                })
        
        return report




In [7]:
"""
Robust Profitability and Efficiency Ratio Calculators
Using FinancialCalculatorBase utilities
"""

import pandas as pd
import numpy as np
from typing import Dict, Any, Optional


# ============================================================================
# PROFITABILITY RATIOS CALCULATOR
# ============================================================================

class RobustProfitabilityCalculator(FinancialCalculatorBase):
    """
    Calculate profitability ratios with comprehensive error handling
    
    Ratios calculated:
    - ROE (Return on Equity)
    - ROA (Return on Assets)
    - Net Profit Margin
    - Gross Profit Margin
    - Operating Margin
    - EBITDA Margin
    """
    
    def __init__(self, ticker: str, income_statement: pd.DataFrame, 
                 balance_sheet: pd.DataFrame):
        """
        Args:
            ticker: Stock ticker
            income_statement: Income statement data
            balance_sheet: Balance sheet data (needed for ROE, ROA)
        """
        # Use income statement as primary data
        super().__init__(ticker, income_statement, min_threshold=1e6, max_ratio=1000.0)
        self.income_statement = income_statement
        self.balance_sheet = balance_sheet

    
    def calculate_roe(self) -> pd.Series:
        # Net income from income statement
        net_income = self.safe_get('Net Income')
        
        # Equity from balance sheet
        if self.balance_sheet is not None and 'Stockholders Equity' in self.balance_sheet.index:
            equity = self.balance_sheet.loc['Stockholders Equity']
            
            # Align indices (income statement and balance sheet may have different dates)
            equity = equity.reindex(net_income.index)
        else:
            print("    ⚠️  Stockholders Equity not available in balance sheet")
            return pd.Series(np.nan, index=net_income.index)
        
        # Calculate with safe division
        roe = self.safe_divide(net_income, equity, strategy='nan', max_value=200)
        roe = roe * 100  # Convert to percentage
        
        # Check for negative equity
        negative_equity_count = (equity < 0).sum()
        if negative_equity_count > 0:
            print(f"    ⚠️  {negative_equity_count} periods with NEGATIVE equity")
            print(f"       → Company may be insolvent")
        
        # Interpolate missing
        roe = self.interpolate_missing(roe, 'ROE', default=15.0)
        
        print(f"    ✓ ROE: {roe.min():.1f}% to {roe.max():.1f}%")
        return roe
    
    
    def calculate_roa(self) -> pd.Series:
        """
        ROA (Return on Assets) = (Net Income / Total Assets) × 100
        
        Measures: How efficiently company uses assets to generate profit
        
        Handling:
        - If Assets = 0: NaN (very unusual)
        - Otherwise: Normal calculation
        """
        print("\n  Calculating ROA (Return on Assets)...")
        
        net_income = self.safe_get('Net Income')
        
        # Assets from balance sheet
        if self.balance_sheet is not None and 'Total Assets' in self.balance_sheet.index:
            assets = self.balance_sheet.loc['Total Assets']
            assets = assets.reindex(net_income.index)
        else:
            print("    ⚠️  Total Assets not available in balance sheet")
            return pd.Series(np.nan, index=net_income.index)
        
        # Calculate with safe division
        roa = self.safe_divide(net_income, assets, strategy='nan', max_value=100)
        roa = roa * 100
        
        # Interpolate missing
        roa = self.interpolate_missing(roa, 'ROA', default=8.0)
        
        print(f"    ✓ ROA: {roa.min():.1f}% to {roa.max():.1f}%")
        return roa
    
    def calculate_net_profit_margin(self) -> pd.Series:
        """
        Net Profit Margin = (Net Income / Total Revenue) × 100
        
        Measures: Percentage of revenue that becomes profit
        """
        print("\n  Calculating Net Profit Margin...")
        
        net_income = self.safe_get('Net Income')
        revenue = self.safe_get('Total Revenue')
        
        # Use 'nan' strategy for zero revenue
        margin = self.safe_divide(net_income, revenue, strategy='nan', max_value=1000)
        margin = margin * 100
        
        # Check for zero revenue
        zero_revenue_count = (revenue.abs() < self.MIN_THRESHOLD).sum()
        if zero_revenue_count > 0:
            print(f"    ⚠️  {zero_revenue_count} periods with zero/near-zero revenue")
        
        margin = self.interpolate_missing(margin, 'Net_Profit_Margin', default=15.0)
        
        print(f"    ✓ Net Profit Margin: {margin.min():.1f}% to {margin.max():.1f}%")
        return margin
    
    def calculate_gross_profit_margin(self) -> pd.Series:
        """
        Gross Profit Margin = (Gross Profit / Total Revenue) × 100
        
        Measures: Profitability after COGS, before operating expenses
        """
        print("\n  Calculating Gross Profit Margin...")
        
        gross_profit = self.safe_get('Gross Profit')
        revenue = self.safe_get('Total Revenue')
        
        margin = self.safe_divide(gross_profit, revenue, strategy='nan', max_value=1000)
        margin = margin * 100
        margin = self.interpolate_missing(margin, 'Gross_Profit_Margin', default=40.0)
        
        print(f"    ✓ Gross Profit Margin: {margin.min():.1f}% to {margin.max():.1f}%")
        return margin
    
    def calculate_operating_margin(self) -> pd.Series:
        """
        Operating Margin = (Operating Income / Total Revenue) × 100
        
        Measures: Profitability from core operations
        """
        print("\n  Calculating Operating Margin...")
        
        operating_income = self.safe_get('Operating Income')
        revenue = self.safe_get('Total Revenue')
        
        margin = self.safe_divide(operating_income, revenue, strategy='nan', max_value=1000)
        margin = margin * 100
        margin = self.interpolate_missing(margin, 'Operating_Margin', default=20.0)
        
        print(f"    ✓ Operating Margin: {margin.min():.1f}% to {margin.max():.1f}%")
        return margin
    
    def calculate_ebitda_margin(self) -> pd.Series:
        """
        EBITDA Margin = (EBITDA / Total Revenue) × 100
        
        Measures: Operating performance before non-cash charges
        
        Note: If EBITDA not available, calculate as:
        EBITDA ≈ Operating Income + Depreciation + Amortization
        """
        print("\n  Calculating EBITDA Margin...")
        
        # Try to get EBITDA directly
        ebitda = self.safe_get('EBITDA')
        revenue = self.safe_get('Total Revenue')
        
        # If EBITDA not available, try to calculate it
        if ebitda.isna().all():
            print("    → EBITDA not available, approximating from Operating Income")
            operating_income = self.safe_get('Operating Income')
            depreciation = self.safe_get('Depreciation And Amortization', default=0)
            ebitda = operating_income + depreciation
        
        margin = self.safe_divide(ebitda, revenue, strategy='nan', max_value=1000)
        margin = margin * 100
        margin = self.interpolate_missing(margin, 'EBITDA_Margin', default=25.0)
        
        print(f"    ✓ EBITDA Margin: {margin.min():.1f}% to {margin.max():.1f}%")
        return margin
    
    def calculate_all_ratios(self) -> Dict[str, pd.Series]:
        """Calculate all profitability ratios"""
        print(f"\n{'='*60}")
        print(f"Profitability Ratios for {self.ticker}")
        print(f"{'='*60}")
        
        ratios = {
            'ROE': self.calculate_roe(),
            'ROA': self.calculate_roa(),
            'Net_Profit_Margin': self.calculate_net_profit_margin(),
            'Gross_Profit_Margin': self.calculate_gross_profit_margin(),
            'Operating_Margin': self.calculate_operating_margin(),
            'EBITDA_Margin': self.calculate_ebitda_margin()
        }
        
        # Validate margin hierarchy and ROE/ROA relationship
        self._validate_profitability_constraints(ratios)
        
        print(f"\n{'='*60}")
        print(f"✓ All profitability ratios calculated")
        print(f"{'='*60}\n")
        
        return ratios
    
    def _validate_profitability_constraints(self, ratios: Dict[str, pd.Series]):
        """
        Validate profitability ratio relationships
        
        Constraints:
        1. Margin hierarchy: Net <= Operating <= Gross (for positive values)
        2. ROA <= ROE (typically, due to leverage effect)
        3. All margins should be reasonable (-100% to 100% typically)
        """
        print("\n  Validating profitability constraints...")
        
        violations = 0
        
        # Constraint 1: Margin hierarchy (only for positive margins)
        for idx in ratios['Net_Profit_Margin'].index:
            gross = ratios['Gross_Profit_Margin'][idx]
            operating = ratios['Operating_Margin'][idx]
            net = ratios['Net_Profit_Margin'][idx]
            
            if pd.notna(gross) and pd.notna(operating) and pd.notna(net):
                if gross > 0 and operating > 0 and net > 0:
                    # Operating shouldn't exceed Gross
                    if operating > gross:
                        ratios['Operating_Margin'][idx] = gross
                        violations += 1
                    # Net shouldn't exceed Operating
                    if net > operating:
                        ratios['Net_Profit_Margin'][idx] = operating
                        violations += 1
        
        # Constraint 2: ROA typically <= ROE (due to financial leverage)
        # Note: This can be violated if company has negative equity
        roa_roe_violations = 0
        for idx in ratios['ROA'].index:
            roa = ratios['ROA'][idx]
            roe = ratios['ROE'][idx]
            
            if pd.notna(roa) and pd.notna(roe):
                if roa > 0 and roe > 0 and roa > roe * 1.5:
                    # ROA significantly higher than ROE is unusual
                    # (Could indicate data issue or negative leverage)
                    roa_roe_violations += 1
        
        if violations > 0:
            print(f"    → Fixed {violations} margin hierarchy violations")
        
        if roa_roe_violations > 0:
            print(f"    ⚠️  {roa_roe_violations} periods where ROA > ROE (check for negative leverage)")
        
        if violations == 0 and roa_roe_violations == 0:
            print(f"    ✓ All constraints satisfied")
    
    def get_data_quality_report(self) -> Dict[str, Any]:
        """Generate data quality report for profitability ratios"""
        report = {
            'ticker': self.ticker,
            'total_periods': len(self.data.columns),
            'issues': []
        }
        
        # Check revenue
        revenue = self.safe_get('Total Revenue')
        zero_revenue = (revenue.abs() < self.MIN_THRESHOLD).sum()
        missing_revenue = revenue.isna().sum()
        
        if zero_revenue > 0:
            report['issues'].append({
                'issue': 'Zero Revenue',
                'count': zero_revenue,
                'severity': 'CRITICAL',
                'action': 'Margins marked as NaN, interpolated'
            })
        
        if missing_revenue > 0:
            report['issues'].append({
                'issue': 'Missing Revenue',
                'count': missing_revenue,
                'severity': 'HIGH',
                'action': 'Values interpolated'
            })
        
        # Check net income (profitability)
        net_income = self.safe_get('Net Income')
        loss_periods = (net_income < 0).sum()
        
        if loss_periods > 0:
            report['issues'].append({
                'issue': 'Net Loss Periods',
                'count': loss_periods,
                'severity': 'MEDIUM',
                'action': 'Negative margins preserved'
            })
        
        # Check equity (for ROE)
        if self.balance_sheet is not None and 'Stockholders Equity' in self.balance_sheet.index:
            equity = self.balance_sheet.loc['Stockholders Equity']
            negative_equity = (equity < 0).sum()
            
            if negative_equity > 0:
                report['issues'].append({
                    'issue': 'Negative Equity (affects ROE)',
                    'count': negative_equity,
                    'severity': 'HIGH',
                    'action': 'ROE marked as NaN for these periods'
                })
        
        return report




In [8]:
# ============================================================================
# EFFICIENCY RATIOS CALCULATOR
# ============================================================================

class RobustEfficiencyCalculator(FinancialCalculatorBase):
    """
    Calculate efficiency/activity ratios with comprehensive error handling
    Ratios calculated:
    - Asset Turnover
    - Inventory Turnover
    - Receivables Turnover
    - Payables Turnover
    - Working Capital Turnover
    - Fixed Asset Turnover
    """
    
    def __init__(self, ticker: str, income_statement: pd.DataFrame, balance_sheet: pd.DataFrame):
        super().__init__(ticker, balance_sheet, min_threshold=1e6, max_ratio=100.0)
        self.income_statement = income_statement
        self.balance_sheet = balance_sheet

    def calculate_asset_turnover(self) -> pd.Series:
        """
        Asset Turnover = Total Revenue / Average Total Assets
        
        Measures: How efficiently company uses assets to generate revenue
        
        Note: Ideally use average assets, but if not available, use period-end
        """
        print("\n  Calculating Asset Turnover...")
        
        # Revenue from income statement
        if 'Total Revenue' in self.income_statement.index:
            revenue = self.income_statement.loc['Total Revenue']
        else:
            print("  Total Revenue not available")
            return pd.Series(np.nan, index=self.data.columns)
        
        # Assets from balance sheet
        assets = self.safe_get('Total Assets')
        
        # Align indices
        revenue = revenue.reindex(assets.index)
        
        # Calculate average assets (current + previous period) / 2
        # If not possible, use period-end assets
        try:
            assets_avg = (assets + assets.shift(1)) / 2
            assets_avg = assets_avg.fillna(assets)  # Use period-end for first period
        except:
            assets_avg = assets
        
        # Calculate with safe division
        turnover = self.safe_divide(revenue, assets_avg, strategy='nan', max_value=50)
        
        # Interpolate missing
        turnover = self.interpolate_missing(turnover, 'Asset_Turnover', default=1.0)
        
        print(f"    ✓ Asset Turnover: {turnover.min():.2f} to {turnover.max():.2f}")
        return turnover
    
    def calculate_inventory_turnover(self) -> pd.Series:
        """
        Inventory Turnover = COGS / Average Inventory
        
        Measures: How quickly company sells inventory
        
        Handling:
        - If Inventory = 0: Set to NaN (service companies have no inventory)
        - Higher is generally better (faster inventory movement)
        """
        print("\n  Calculating Inventory Turnover...")
        
        # COGS from income statement
        if 'Cost Of Revenue' in self.income_statement.index:
            cogs = self.income_statement.loc['Cost Of Revenue']
        else:
            print(" COGS not available, using alternative calculation")
            # Try to derive: COGS = Revenue - Gross Profit
            revenue = self.income_statement.loc['Total Revenue'] if 'Total Revenue' in self.income_statement.index else pd.Series(np.nan, index=self.data.columns)
            gross_profit = self.income_statement.loc['Gross Profit'] if 'Gross Profit' in self.income_statement.index else pd.Series(np.nan, index=self.data.columns)
            cogs = revenue - gross_profit
        
        # Inventory from balance sheet
        inventory = self.safe_get('Inventory', default=0)
        
        # Align indices
        cogs = cogs.reindex(inventory.index)
        
        # Check if company has inventory
        has_inventory = (inventory.abs() >= self.MIN_THRESHOLD).any()
        
        if not has_inventory:
            print("    → Company appears to have no inventory (service company)")
            print("       Setting Inventory Turnover to NaN")
            return pd.Series(np.nan, index=inventory.index)
        
        # Calculate average inventory
        try:
            inventory_avg = (inventory + inventory.shift(1)) / 2
            inventory_avg = inventory_avg.fillna(inventory)
        except:
            inventory_avg = inventory
        
        # Calculate with safe division
        turnover = self.safe_divide(cogs, inventory_avg, strategy='nan', max_value=1000)
        
        # Interpolate missing
        turnover = self.interpolate_missing(turnover, 'Inventory_Turnover', default=8.0)
        
        print(f" Inventory Turnover: {turnover.min():.2f} to {turnover.max():.2f}")
        return turnover
    
    def calculate_receivables_turnover(self) -> pd.Series:
        """
        Receivables Turnover = Total Revenue / Average Accounts Receivable
        
        Measures: How quickly company collects payments from customers
        
        Handling:
        - If Receivables = 0: Company operates on cash basis (rare)
        - Higher is generally better (faster collection)
        """
        print("\n  Calculating Receivables Turnover...")
        
        # Revenue from income statement
        if 'Total Revenue' in self.income_statement.index:
            revenue = self.income_statement.loc['Total Revenue']
        else:
            print("    ⚠️  Total Revenue not available")
            return pd.Series(np.nan, index=self.data.columns)
        
        # Receivables from balance sheet
        receivables = self.safe_get('Accounts Receivable', default=0)
        
        # Align indices
        revenue = revenue.reindex(receivables.index)
        
        # Check if company has receivables
        has_receivables = (receivables.abs() >= self.MIN_THRESHOLD).any()
        
        if not has_receivables:
            print("    → Company has minimal receivables (cash-based business)")
            print("       Setting Receivables Turnover to NaN")
            return pd.Series(np.nan, index=receivables.index)
        
        # Calculate average receivables
        try:
            receivables_avg = (receivables + receivables.shift(1)) / 2
            receivables_avg = receivables_avg.fillna(receivables)
        except:
            receivables_avg = receivables
        
        # Calculate with safe division
        turnover = self.safe_divide(revenue, receivables_avg, strategy='nan', max_value=1000)
        
        # Interpolate missing
        turnover = self.interpolate_missing(turnover, 'Receivables_Turnover', default=12.0)
        
        print(f"    ✓ Receivables Turnover: {turnover.min():.2f} to {turnover.max():.2f}")
        return turnover
    
    def calculate_payables_turnover(self) -> pd.Series:
        """
        Payables Turnover = COGS / Average Accounts Payable
        
        Measures: How quickly company pays suppliers
        
        Note: Lower can be better (retaining cash longer)
        """
        print("\n  Calculating Payables Turnover...")
        
        # COGS from income statement
        if 'Cost Of Revenue' in self.income_statement.index:
            cogs = self.income_statement.loc['Cost Of Revenue']
        else:
            revenue = self.income_statement.loc['Total Revenue'] if 'Total Revenue' in self.income_statement.index else pd.Series(np.nan, index=self.data.columns)
            gross_profit = self.income_statement.loc['Gross Profit'] if 'Gross Profit' in self.income_statement.index else pd.Series(np.nan, index=self.data.columns)
            cogs = revenue - gross_profit
        
        # Payables from balance sheet
        payables = self.safe_get('Accounts Payable', default=0)
        
        # Align indices
        cogs = cogs.reindex(payables.index)
        
        # Calculate average payables
        try:
            payables_avg = (payables + payables.shift(1)) / 2
            payables_avg = payables_avg.fillna(payables)
        except:
            payables_avg = payables
        
        # Calculate with safe division
        turnover = self.safe_divide(cogs, payables_avg, strategy='nan', max_value=1000)
        
        # Interpolate missing
        turnover = self.interpolate_missing(turnover, 'Payables_Turnover', default=10.0)
        
        print(f"    ✓ Payables Turnover: {turnover.min():.2f} to {turnover.max():.2f}")
        return turnover
    
    def calculate_working_capital_turnover(self) -> pd.Series:
        """
        Working Capital Turnover = Total Revenue / Average Working Capital
        
        Measures: How efficiently company uses working capital
        
        Handling:
        - If Working Capital = 0 or negative: Mark as NaN
        - Higher generally better (more revenue per dollar of working capital)
        """
        print("\n  Calculating Working Capital Turnover...")
        
        # Revenue from income statement
        if 'Total Revenue' in self.income_statement.index:
            revenue = self.income_statement.loc['Total Revenue']
        else:
            print("    ⚠️  Total Revenue not available")
            return pd.Series(np.nan, index=self.data.columns)
        
        # Working Capital = Current Assets - Current Liabilities
        current_assets = self.safe_get('Current Assets')
        current_liabilities = self.safe_get('Current Liabilities')
        working_capital = current_assets - current_liabilities
        
        # Align indices
        revenue = revenue.reindex(working_capital.index)
        
        # Calculate average working capital
        try:
            wc_avg = (working_capital + working_capital.shift(1)) / 2
            wc_avg = wc_avg.fillna(working_capital)
        except:
            wc_avg = working_capital
        
        # Calculate with safe division
        turnover = self.safe_divide(revenue, wc_avg, strategy='nan', max_value=100)
        
        # Interpolate missing
        turnover = self.interpolate_missing(turnover, 'Working_Capital_Turnover', default=5.0)
        
        print(f"    ✓ Working Capital Turnover: {turnover.min():.2f} to {turnover.max():.2f}")
        return turnover
    
    def calculate_fixed_asset_turnover(self) -> pd.Series:
        """
        Fixed Asset Turnover = Total Revenue / Average Fixed Assets (PP&E)
        
        Measures: How efficiently company uses fixed assets to generate revenue
        
        Note: Higher is generally better (more revenue per dollar of PP&E)
        """
        print("\n  Calculating Fixed Asset Turnover...")
        
        # Revenue from income statement
        if 'Total Revenue' in self.income_statement.index:
            revenue = self.income_statement.loc['Total Revenue']
        else:
            print("    ⚠️  Total Revenue not available")
            return pd.Series(np.nan, index=self.data.columns)
        
        # PP&E from balance sheet
        ppe = self.safe_get('Net PPE', default=0)
        
        # If Net PPE not available, try alternatives
        if ppe.isna().all() or (ppe.abs() < self.MIN_THRESHOLD).all():
            ppe = self.safe_get('Property Plant Equipment', default=0)
        
        # Align indices
        revenue = revenue.reindex(ppe.index)
        
        # Calculate average PP&E
        try:
            ppe_avg = (ppe + ppe.shift(1)) / 2
            ppe_avg = ppe_avg.fillna(ppe)
        except:
            ppe_avg = ppe
        
        # Calculate with safe division
        turnover = self.safe_divide(revenue, ppe_avg, strategy='nan', max_value=100)
        
        # Interpolate missing
        turnover = self.interpolate_missing(turnover, 'Fixed_Asset_Turnover', default=3.0)
        
        print(f"    ✓ Fixed Asset Turnover: {turnover.min():.2f} to {turnover.max():.2f}")
        return turnover
    
    def calculate_all_ratios(self) -> Dict[str, pd.Series]:
        """Calculate all efficiency ratios"""
        print(f"\n{'='*60}")
        print(f"Efficiency Ratios for {self.ticker}")
        print(f"{'='*60}")
        
        ratios = {
            'Asset_Turnover': self.calculate_asset_turnover(),
            'Inventory_Turnover': self.calculate_inventory_turnover(),
            'Receivables_Turnover': self.calculate_receivables_turnover(),
            'Payables_Turnover': self.calculate_payables_turnover(),
            'Working_Capital_Turnover': self.calculate_working_capital_turnover(),
            'Fixed_Asset_Turnover': self.calculate_fixed_asset_turnover()
        }
        
        # Validate efficiency ratios
        self._validate_efficiency_constraints(ratios)
        self.get_data_quality_report()
        
        print(f"\n{'='*60}")
        print(f"✓ All efficiency ratios calculated")
        print(f"{'='*60}\n")
        
        return ratios
    
    def _validate_efficiency_constraints(self, ratios: Dict[str, pd.Series]):
        """
        Validate efficiency ratio relationships
        
        Checks:
        1. All turnover ratios should be positive
        2. Extreme values capped
        3. Receivables Turnover × Days ≈ 365 (days sales outstanding)
        """
        print("\n  Validating efficiency constraints...")
        
        violations = 0
        
        # Ensure all ratios are non-negative
        for ratio_name, ratio_series in ratios.items():
            negative_count = (ratio_series < 0).sum()
            if negative_count > 0:
                print(f"{ratio_name}: {negative_count} negative values found")
                ratios[ratio_name] = ratio_series.clip(lower=0)
                violations += negative_count
        
        if violations > 0:
            print(f"    → Fixed {violations} negative ratio values")
        else:
            print(f"    ✓ All efficiency ratios valid")
    
    def get_data_quality_report(self) -> Dict[str, Any]:
        """Generate data quality report for efficiency ratios"""
        report = {
            'ticker': self.ticker,
            'total_periods': len(self.data.columns),
            'issues': []
        }
        
        # Check revenue
        if 'Total Revenue' in self.income_statement.index:
            revenue = self.income_statement.loc['Total Revenue']
            zero_revenue = (revenue.abs() < self.MIN_THRESHOLD).sum()
            
            if zero_revenue > 0:
                report['issues'].append({
                    'issue': 'Zero Revenue (affects turnover ratios)',
                    'count': zero_revenue,
                    'severity': 'HIGH',
                    'action': 'Turnover ratios marked as NaN'
                })
        
        # Check inventory
        inventory = self.safe_get('Inventory')
        no_inventory = (inventory.abs() < self.MIN_THRESHOLD).all()
        
        if no_inventory:
            report['issues'].append({
                'issue': 'No Inventory (service company)',
                'count': len(self.data.columns),
                'severity': 'INFO',
                'action': 'Inventory Turnover set to NaN (expected)'
            })
        
        # Check receivables
        receivables = self.safe_get('Accounts Receivable')
        no_receivables = (receivables.abs() < self.MIN_THRESHOLD).all()
        
        if no_receivables:
            report['issues'].append({
                'issue': 'No Receivables (cash business)',
                'count': len(self.data.columns),
                'severity': 'INFO',
                'action': 'Receivables Turnover set to NaN (expected)'
            })

        print ( report)
        return report




In [15]:
"""
Robust Growth Metrics Calculator
Handles zero values, negative values, and extreme growth rates
"""

# ============================================================================
# GROWTH METRICS CALCULATOR
# ============================================================================

class RobustGrowthCalculator(FinancialCalculatorBase):
    """
    Calculate growth metrics with comprehensive error handling
    
    Metrics calculated:
    - Revenue Growth (QoQ, YoY)
    - Net Income Growth (QoQ, YoY)
    - Assets Growth (QoQ, YoY)
    - Equity Growth (QoQ, YoY)
    - EPS Growth (QoQ, YoY)
    - Operating Income Growth (QoQ, YoY)
    
    Handles:
    - Zero to positive transitions (growth = inf → capped)
    - Negative to positive transitions (special handling)
    - Extreme growth rates (capped at ±1000%)
    - Missing data (interpolation)
    """
 
    def __init__(self, ticker: str, income_statement: pd.DataFrame,
                 balance_sheet: pd.DataFrame):
        """
        Args:
            ticker: Stock ticker
            income_statement: Income statement data
            balance_sheet: Balance sheet data
        """
        super().__init__(ticker, income_statement, min_threshold=1e6, max_ratio=1000.0)
        self.income_statement = income_statement
        self.balance_sheet = balance_sheet
        
        # Growth-specific configuration
        self.MAX_GROWTH = 1000.0  # Cap growth at ±1000%
        self.MIN_BASE_VALUE = 1e3  # $1K minimum for valid growth calc
    
    def calculate_growth_rate(self, series: pd.Series, periods: int = 1,
                              metric_name: str = "metric") -> pd.Series:
        """
        Calculate growth rate with robust error handling
        
        Args:
            series: Time series data
            periods: Number of periods to look back
                    -1 = QoQ (quarter-over-quarter)
                    -4 = YoY (year-over-year)
            metric_name: Name of metric for logging
        
        Returns:
            Growth rate in percentage
        
        Formula:
            Growth% = ((Current - Previous) / |Previous|) × 100
        
        Handling:
        1. Zero/near-zero previous value → NaN (can't calculate meaningful growth)
        2. Negative to positive → Special "recovery" signal
        3. Positive to negative → Negative growth
        4. Extreme values → Capped at MAX_GROWTH
        """
        growth = pd.Series(index=series.index, dtype=float)
        
        for i in range(len(series)):
            # Get current and previous values
            if i + periods >= len(series) or i + periods < 0:
                # Not enough data for this period
                growth.iloc[i] = np.nan
                continue
            
            current = series.iloc[i]
            previous = series.iloc[i + periods]  # periods is negative for QoQ
            
            # Handle missing values
            if pd.isna(current) or pd.isna(previous):
                growth.iloc[i] = np.nan
                continue
            
            # Case 1: Previous value is zero/near-zero
            if abs(previous) < self.MIN_BASE_VALUE:
                if abs(current) < self.MIN_BASE_VALUE:
                    # Both zero → no growth
                    growth.iloc[i] = 0.0
                else:
                    # Zero to non-zero → can't calculate meaningful %
                    # Mark as NaN for interpolation
                    growth.iloc[i] = np.nan
                continue
            
            # Case 2: Normal growth calculation
            growth_rate = ((current - previous) / abs(previous)) * 100
            
            # Cap extreme values
            if abs(growth_rate) > self.MAX_GROWTH:
                growth.iloc[i] = self.MAX_GROWTH if growth_rate > 0 else -self.MAX_GROWTH
            else:
                growth.iloc[i] = growth_rate
        
        return growth
    
    def calculate_revenue_growth_qoq(self) -> pd.Series:
        """
        Revenue Growth (Quarter-over-Quarter)
        
        Formula: ((Current Q Revenue - Previous Q Revenue) / Previous Q Revenue) × 100
        """
        print("\n  Calculating Revenue Growth (QoQ)...")
        
        if 'Total Revenue' not in self.income_statement.index:
            print("    ⚠️  Total Revenue not available")
            return pd.Series(np.nan, index=self.income_statement.columns)
        
        revenue = self.income_statement.loc['Total Revenue']
        
        # Calculate growth (periods=-1 means compare with previous period)
        growth = self.calculate_growth_rate(revenue, periods=-1, metric_name="Revenue")
        
        # Check for issues
        zero_revenue_count = (revenue.abs() < self.MIN_BASE_VALUE).sum()
        if zero_revenue_count > 0:
            print(f"    ⚠️  {zero_revenue_count} periods with zero/near-zero revenue")
        
        # Interpolate missing
        growth = self.interpolate_missing(growth, 'Revenue_Growth_QoQ', default=0.0)
        
        if not growth.isna().all():
            print(f"    ✓ Revenue Growth (QoQ): {growth.min():.1f}% to {growth.max():.1f}%")
        
        return growth
    
    def calculate_revenue_growth_yoy(self) -> pd.Series:
        """
        Revenue Growth (Year-over-Year)
        
        Formula: ((Current Q Revenue - Same Q Last Year Revenue) / Same Q Last Year Revenue) × 100
        
        Note: Assumes quarterly data, so periods=-4
        """
        print("\n  Calculating Revenue Growth (YoY)...")
        
        if 'Total Revenue' not in self.income_statement.index:
            print("    ⚠️  Total Revenue not available")
            return pd.Series(np.nan, index=self.income_statement.columns)
        
        revenue = self.income_statement.loc['Total Revenue']
        
        # Calculate YoY growth (periods=-4 for quarterly data)
        growth = self.calculate_growth_rate(revenue, periods=-4, metric_name="Revenue YoY")
        
        # Interpolate missing
        growth = self.interpolate_missing(growth, 'Revenue_Growth_YoY', default=0.0)
        
        if not growth.isna().all():
            print(f"    ✓ Revenue Growth (YoY): {growth.min():.1f}% to {growth.max():.1f}%")
        
        return growth
    
    def calculate_net_income_growth_qoq(self) -> pd.Series:
        """
        Net Income Growth (Quarter-over-Quarter)
        
        Special handling for losses:
        - Loss to bigger loss = negative growth
        - Loss to smaller loss = positive growth (improvement)
        - Loss to profit = recovery (capped growth)
        """
        print("\n  Calculating Net Income Growth (QoQ)...")
        
        if 'Net Income' not in self.income_statement.index:
            print(" Net Income not available")
            return pd.Series(np.nan, index=self.income_statement.columns)
        
        net_income = self.income_statement.loc['Net Income']
        
        # Calculate growth
        growth = self.calculate_growth_rate(net_income, periods=-1, metric_name="Net Income")
        
        # Check for loss periods
        loss_count = (net_income < 0).sum()
        if loss_count > 0:
            print(f" {loss_count} periods with net losses")
            print(f"       → Growth rates may be volatile")
        
        # Interpolate missing
        growth = self.interpolate_missing(growth, 'Net_Income_Growth_QoQ', default=0.0)
        
        if not growth.isna().all():
            print(f"    ✓ Net Income Growth (QoQ): {growth.min():.1f}% to {growth.max():.1f}%")
        
        return growth
    
    def calculate_net_income_growth_yoy(self) -> pd.Series:
        """Net Income Growth (Year-over-Year)"""
        print("\n  Calculating Net Income Growth (YoY)...")
        
        if 'Net Income' not in self.income_statement.index:
            print(" Net Income not available")
            return pd.Series(np.nan, index=self.income_statement.columns)
        
        net_income = self.income_statement.loc['Net Income']
        growth = self.calculate_growth_rate(net_income, periods=-4, metric_name="Net Income YoY")
        growth = self.interpolate_missing(growth, 'Net_Income_Growth_YoY', default=0.0)
        
        if not growth.isna().all():
            print(f"    ✓ Net Income Growth (YoY): {growth.min():.1f}% to {growth.max():.1f}%")
        
        return growth
    
    def calculate_assets_growth_qoq(self) -> pd.Series:
        """Total Assets Growth (Quarter-over-Quarter)"""
        print("\n  Calculating Assets Growth (QoQ)...")
        
        if self.balance_sheet is None or 'Total Assets' not in self.balance_sheet.index:
            print("   Total Assets not available")
            return pd.Series(np.nan, index=self.income_statement.columns)
        
        assets = self.balance_sheet.loc['Total Assets']
        
        # Align with income statement index
        assets = assets.reindex(self.income_statement.columns)
        
        # Calculate growth
        growth = self.calculate_growth_rate(assets, periods=-1, metric_name="Assets")
        growth = self.interpolate_missing(growth, 'Assets_Growth_QoQ', default=0.0)
        
        if not growth.isna().all():
            print(f"    ✓ Assets Growth (QoQ): {growth.min():.1f}% to {growth.max():.1f}%")
        
        return growth
    
    def calculate_assets_growth_yoy(self) -> pd.Series:
        """Total Assets Growth (Year-over-Year)"""
        print("\n  Calculating Assets Growth (YoY)...")
        
        if self.balance_sheet is None or 'Total Assets' not in self.balance_sheet.index:
            print("    ⚠️  Total Assets not available")
            return pd.Series(np.nan, index=self.income_statement.columns)
        
        assets = self.balance_sheet.loc['Total Assets']
        assets = assets.reindex(self.income_statement.columns)
        growth = self.calculate_growth_rate(assets, periods=-4, metric_name="Assets YoY")
        growth = self.interpolate_missing(growth, 'Assets_Growth_YoY', default=0.0)
        
        if not growth.isna().all():
            print(f"    ✓ Assets Growth (YoY): {growth.min():.1f}% to {growth.max():.1f}%")
        
        return growth
    
    def calculate_equity_growth_qoq(self) -> pd.Series:
        """Stockholders Equity Growth (Quarter-over-Quarter)"""
        print("\n  Calculating Equity Growth (QoQ)...")
        
        if self.balance_sheet is None or 'Stockholders Equity' not in self.balance_sheet.index:
            print("    ⚠️  Stockholders Equity not available")
            return pd.Series(np.nan, index=self.income_statement.columns)
        
        equity = self.balance_sheet.loc['Stockholders Equity']
        equity = equity.reindex(self.income_statement.columns)
        
        # Check for negative equity
        negative_equity_count = (equity < 0).sum()
        if negative_equity_count > 0:
            print(f"    ⚠️  {negative_equity_count} periods with negative equity")
        
        growth = self.calculate_growth_rate(equity, periods=-1, metric_name="Equity")
        growth = self.interpolate_missing(growth, 'Equity_Growth_QoQ', default=0.0)
        
        if not growth.isna().all():
            print(f"    ✓ Equity Growth (QoQ): {growth.min():.1f}% to {growth.max():.1f}%")
        
        return growth
    
    def calculate_equity_growth_yoy(self) -> pd.Series:
        """Stockholders Equity Growth (Year-over-Year)"""
        print("\n  Calculating Equity Growth (YoY)...")
        
        if self.balance_sheet is None or 'Stockholders Equity' not in self.balance_sheet.index:
            print("    ⚠️  Stockholders Equity not available")
            return pd.Series(np.nan, index=self.income_statement.columns)
        
        equity = self.balance_sheet.loc['Stockholders Equity']
        equity = equity.reindex(self.income_statement.columns)
        growth = self.calculate_growth_rate(equity, periods=-4, metric_name="Equity YoY")
        growth = self.interpolate_missing(growth, 'Equity_Growth_YoY', default=0.0)
        
        if not growth.isna().all():
            print(f"    ✓ Equity Growth (YoY): {growth.min():.1f}% to {growth.max():.1f}%")
        
        return growth
    
    def calculate_eps_growth_qoq(self) -> pd.Series:
        """
        EPS Growth (Quarter-over-Quarter)
        
        Note: If EPS not available, can calculate as Net Income / Shares Outstanding
        """
        print("\n  Calculating EPS Growth (QoQ)...")
        
        if 'Basic EPS' in self.income_statement.index:
            eps = self.income_statement.loc['Basic EPS']
        elif 'Diluted EPS' in self.income_statement.index:
            eps = self.income_statement.loc['Diluted EPS']
        else:
            print(" EPS not available")
            return pd.Series(np.nan, index=self.income_statement.columns)
        print(" EPS Value", eps)
        series = eps
        #growth = self.calculate_growth_rate(eps, periods=-1, metric_name="EPS")
        #growth = self.interpolate_missing(growth, 'EPS_Growth_QoQ', default=0.0)
        periods = 1
        series = series.sort_index()
        print( series)
        
        growth = series.pct_change(periods=abs(periods)) * 100
        print( growth)
        # Handle near-zero base
        previous = series.shift(abs(periods)) #.abs() < self.MIN_BASE_VALUE
        mask = previous.abs() < 1e-6
        print (mask)
        growth[mask] = np.nan       
        # Cap extreme values
        growth = growth.clip(-self.MAX_GROWTH, self.MAX_GROWTH)
        print( "eps growth", growth)
        growth = self.interpolate_missing(growth, 'EPS_Growth_QoQ', default=0.0)
        if not growth.isna().all():
            print(f"    ✓ EPS Growth (QoQ): {growth.min():.1f}% to {growth.max():.1f}%")       
        return growth
        
    
    def calculate_operating_income_growth_qoq(self) -> pd.Series:
        """Operating Income Growth (Quarter-over-Quarter)"""
        print("\n  Calculating Operating Income Growth (QoQ)...")
        
        if 'Operating Income' not in self.income_statement.index:
            print("    ⚠️  Operating Income not available")
            return pd.Series(np.nan, index=self.income_statement.columns)
        
        operating_income = self.income_statement.loc['Operating Income']
        growth = self.calculate_growth_rate(operating_income, periods=-1, 
                                           metric_name="Operating Income")
        growth = self.interpolate_missing(growth, 'Operating_Income_Growth_QoQ', default=0.0)
        
        if not growth.isna().all():
            print(f"    ✓ Operating Income Growth (QoQ): {growth.min():.1f}% to {growth.max():.1f}%")
        
        return growth
    
    def calculate_all_ratios(self) -> Dict[str, pd.Series]:
        """Calculate all growth metrics (QoQ and YoY)"""
        print(f"\n{'='*60}")
        print(f"Growth Metrics for {self.ticker}")
        print(f"{'='*60}")
        if self.data is None or self.data.empty:
            print(f"Skipping {self.ticker}: no income statement data available")
            return None
            
        growth_metrics = {
            # Quarter-over-Quarter
            'Revenue_Growth_QoQ': self.calculate_revenue_growth_qoq(),
            'Net_Income_Growth_QoQ': self.calculate_net_income_growth_qoq(),
            'Assets_Growth_QoQ': self.calculate_assets_growth_qoq(),
            'Equity_Growth_QoQ': self.calculate_equity_growth_qoq(),
            'EPS_Growth_QoQ': self.calculate_eps_growth_qoq(),
            'Operating_Income_Growth_QoQ': self.calculate_operating_income_growth_qoq(),
            
            # Year-over-Year
            'Revenue_Growth_YoY': self.calculate_revenue_growth_yoy(),
            'Net_Income_Growth_YoY': self.calculate_net_income_growth_yoy(),
            'Assets_Growth_YoY': self.calculate_assets_growth_yoy(),
            'Equity_Growth_YoY': self.calculate_equity_growth_yoy(),
        }
        
        # Validate growth metrics
        self._validate_growth_metrics(growth_metrics)

        self.get_data_quality_report()
        
        print(f"\n{'='*60}")
        print(f"✓ All growth metrics calculated")
        print(f"{'='*60}\n")
        
        return growth_metrics
    
    def _validate_growth_metrics(self, metrics: Dict[str, pd.Series]):
        """
        Validate growth metrics
        
        Checks:
        1. All growth rates capped at ±MAX_GROWTH
        2. YoY growth should generally be smoother than QoQ
        3. Warn about extreme volatility
        """
        print("\n  Validating growth metrics...")
        
        extreme_count = 0
        
        for name, series in metrics.items():
            # Check for extreme values
            extreme = (series.abs() > self.MAX_GROWTH * 0.8).sum()
            if extreme > 0:
                extreme_count += extreme
        
        if extreme_count > 0:
            print(f" {extreme_count} periods with extreme growth rates (>800%)")
            print(f" Values capped at ±{self.MAX_GROWTH}%")
        else:
            print(f"  All growth metrics within reasonable bounds")
    
    def get_data_quality_report(self) -> Dict[str, Any]:
        """Generate data quality report for growth metrics"""
        report = {
            'ticker': self.ticker,
            'total_periods': len(self.income_statement.columns),
            'issues': []
        }
        
        # Check revenue
        if 'Total Revenue' in self.income_statement.index:
            revenue = self.income_statement.loc['Total Revenue']
            zero_revenue = (revenue.abs() < self.MIN_BASE_VALUE).sum()
            
            if zero_revenue > 0:
                report['issues'].append({
                    'issue': 'Zero/Near-Zero Revenue',
                    'count': zero_revenue,
                    'severity': 'HIGH',
                    'action': 'Growth rates marked as NaN, interpolated'
                })
        
        # Check net income volatility
        if 'Net Income' in self.income_statement.index:
            net_income = self.income_statement.loc['Net Income']
            sign_changes = ((net_income > 0) != (net_income.shift(-1) > 0)).sum()
            
            if sign_changes > len(net_income) / 2:
                report['issues'].append({
                    'issue': 'High Net Income Volatility',
                    'count': sign_changes,
                    'severity': 'MEDIUM',
                    'action': 'Profit/loss alternating - growth rates may be extreme'
                })
        
        # Check for insufficient data
        if len(self.income_statement.columns) < 4:
            report['issues'].append({
                'issue': 'Insufficient Data for YoY',
                'count': len(self.income_statement.columns),
                'severity': 'INFO',
                'action': 'YoY growth metrics will have many NaN values'
            })

        print ( report)
        return report


In [16]:
# =============================================================================
# CLASS 2: FEATURE ENGINEER - Derives new columns and calculates ratios
# =============================================================================

class FinancialFeatureEngineer:
    """Calculates financial ratios and derives new features"""
    
    def __init__(self, ticker, sector, balance_sheet=None, income_statement=None, cash_flow=None):
        self.ticker = ticker
        self.sector = sector
        self.balance_sheet = balance_sheet
        self.income_statement = income_statement
        self.cash_flow = cash_flow
        self.features = pd.DataFrame()
        self.MAX_RATIO = 100.0  # Cap ratios at this value
        self.MIN_LIABILITY_THRESHOLD = 1e-6  # Treat as zero if below this

        self.liquidity_calc = RobustLiquidityCalculator( ticker, balance_sheet)
        # Use robust calculator
        self.leverage_calc = RobustLeverageCalculator(ticker, balance_sheet)
        self.pro_calc = RobustProfitabilityCalculator(ticker, income_statement, balance_sheet)
        self.eff_calc = RobustEfficiencyCalculator(ticker, income_statement, balance_sheet)
        self.growth_calc  = RobustGrowthCalculator(ticker, income_statement, balance_sheet)
    

    def calculate_all_features(self):
        """Calculate all financial ratios and features"""
        print(f"\n{'='*60}")
        print("FEATURE ENGINEERING: Calculating All Financial Ratios")
        print(f"{'='*60}")
        
        all_features = []

        '''
        self.leverage_calc = RobustLeverageCalculator(ticker, balance_sheet)
        self.eff_calc = RobustProfitabilityCalculator(ticker, balance_sheet)
        self.pro_calc = RobustEfficiencyCalculator(ticker, balance_sheet)
        '''
        
        # Calculate all ratio categories
        liquidity = self.liquidity_calc.calculate_all_liquidity_ratios()
        print( liquidity)
        if liquidity is not None:
            all_features.append(liquidity)
        self.liquidity_calc.get_data_quality_report()
        
        leverage = self.leverage_calc.calculate_all_leverage_ratios()
        print( leverage)
        if leverage is not None:
            all_features.append(leverage)
        quality_report = self.liquidity_calc.get_data_quality_report()
        
        profitability = self.pro_calc.calculate_all_ratios()
        print( profitability)
        if profitability is not None:
            all_features.append(profitability)
        
        efficiency = self.eff_calc.calculate_all_ratios()
        print( efficiency)
        if efficiency is not None:
            all_features.append(efficiency)
        
        growth = self.growth_calc.calculate_all_ratios()
        print( growth)
        if growth is not None:
            all_features.append(growth)

        #df_sector = self.sector_engine.add_all_features(df)
        # View the features
        #print("\nDataFrame shape:", df_with_features.shape)
        #print("\nFeature columns:")
        #print(self.sector_engine.get_feature_names())

        print( "dict: all features ",all_features )
        combined = {}
        for d in all_features:
            combined.update(d)
        # Combine all features
        if all_features:
            self.features = df = pd.DataFrame(combined)# pd.concat(all_features, axis=1)
            print( "get features", self.features.head(5))
            self.features = self.features.dropna(axis=0, how='all')
            print( "get features", self.features.head(5))
            self.features['Sector'] = self.sector
            print(f"\n✓ Total features calculated: {len(self.features.columns)}")
            """
            df.insert(0, 'Ticker', self.ticker)
            df.insert(1, 'Company_Name', self.sector_info['company_name'])
            df.insert(2, 'Sector', self.sector_info['sector'])
            df.insert(3, 'Industry', self.sector_info['industry'])
            df.insert(4, 'Country', self.sector_info['country'])
            df.insert(5, 'Market_Cap', self.sector_info['market_cap'])
            """
            #df_stock['Date'] = pd.to_datetime(df_stock['Date'])
            #df_sector['date'] = pd.to_datetime(df_sector['date'])
            
            #merged_df = pd.merge(df, df_sector, on=['Date', 'Ticker'], how='left')
            #merged_df = merged_df.sort_values(['Ticker', 'Date'])
            #merged_df = merged_df.groupby('Ticker').ffill()
            return self.features
        else:
            print("\n✗ No features could be calculated")
            return None
    
    def get_feature_summary(self):
        """Get summary statistics of all calculated features"""
        if self.features.empty:
            print("No features available. Run calculate_all_features() first.")
            return None
        
        print(f"\n{'='*60}")
        print("FEATURE SUMMARY")
        print(f"{'='*60}\n")
        
        summary = self.features.T.describe()
        #print(summary)
        return summary

    def get_sector_summary(self) -> Dict:
        """Get summary of sector information"""
        return {
            'ticker': self.ticker,
            'company': self.sector_info['company_name'],
            'sector': self.sector_info['sector'],
            'industry': self.sector_info['industry'],
            'market_cap_billions': self.sector_info['market_cap'] / 1e9 if self.sector_info['market_cap'] else None,
            'employees': self.sector_info['employees']
        }


In [17]:
# =============================================================================
# CLASS 3: ANALYZER - Combines data and analyzes correlations
# =============================================================================

class StockFinancialAnalyzer:
    """Main analyzer that combines data fetching and feature engineering"""
    
    def __init__(self, ticker):
        self.ticker = ticker
        self.fetcher = StockDataFetcher(ticker)
        self.engineer = None
        self.data = {}
        self.merged_data = None
        
    def fetch_and_prepare_data(self):
        """Fetch all data and prepare for analysis"""
        # Fetch all financial data
        self.data = self.fetcher.fetch_all_data()
        self.sector_data = self.fetcher.get_sector_info()
        print( "Sector Info", self.sector_data)
        # Initialize feature engineer
        self.engineer = FinancialFeatureEngineer( ticker,
                                                  self.sector_data,
                                                  balance_sheet=self.data['balance_sheet'],
                                                  income_statement=self.data['income_statement'],
                                                  cash_flow=self.data['cash_flow']
                                                )

        # Calculate all features
        features = self.engineer.calculate_all_features()
        
        print ("features", features.head())
        return features
    
    def merge_with_stock_prices(self):
        """Merge financial features with stock prices"""
        if self.engineer is None or self.engineer.features.empty:
            print(" No   available. Run fetch_and_prepare_data() first.")
            return None
        
        print(f"\n{'='*60}")
        print("Merging Features with Stock Prices")
        print(f"{'='*60}\n")
        
        stock_prices = self.data['stock_prices']
        features = self.engineer.features
        features = features.dropna(axis=1)
        print ( stock_prices.head(5))
        print ( features.head(5))
        
        merged = []

        print ( "features df")
        print (features.columns)
        
        #stock_prices.index = pd.to_datetime(stock_prices.index)
          # Convert index to datetime if not already
        if not isinstance(features.index, pd.DatetimeIndex):
            features.index = pd.to_datetime(features.index)
        
        if not isinstance(stock_prices.index, pd.DatetimeIndex):
            stock_prices.index = pd.to_datetime(stock_prices.index)
            
        stock_prices.index = stock_prices.index.tz_localize(None).normalize()
        features.index = features.index.tz_localize(None).normalize()
        
        print ("update index format", stock_prices.head(5))
        print ( type(stock_prices.index))
        print ( type(features.index))
        
        # For each date in features (financial statement dates)
        for date in features.index:
            print ( type(date), date)

            closest_date = stock_prices.index[
                stock_prices.index.get_indexer([date], method='nearest')[0]
            ]
            
            price = stock_prices.loc[closest_date, 'Close']
            
            # Create row with date, price, and all features
            row = {'Date': date, 'Stock_Price': price}
            
            # Add all feature values for this date
            for feature_name in features.columns:
                row[feature_name] = features.loc[date, feature_name]
            
            merged.append(row)
        
        self.merged_data = pd.DataFrame(merged).sort_values('Date')

        print( self.merged_data.head(5))
        print(f"✓ Merged {len(self.merged_data)} periods of data")
        print(f"✓ Total columns: {len(self.merged_data.columns)}")
        print(f"✓ Size: {(self.merged_data.shape)}")
        return self.merged_data
    
    def analyze_correlations(self, top_n=10):
        """Analyze correlation between features and stock price"""
        if self.merged_data is None:
            print("  No merged data available. Run merge_with_stock_prices() first.")
            return None
        
        print(f"\n{'='*60}")
        print("CORRELATION ANALYSIS")
        print(f"{'='*60}\n")
        
        # Calculate correlations with stock price
        numeric_cols = self.merged_data.select_dtypes(include=[np.number]).columns
        feature_cols = [col for col in numeric_cols if col != 'Stock_Price']
        
        correlations = {}
        for col in feature_cols:
            # Drop NaN values for correlation calculation
            valid_data = self.merged_data[[col, 'Stock_Price']].dropna()
            if len(valid_data) > 1:
                corr = valid_data[col].corr(valid_data['Stock_Price'])
                correlations[col] = corr
        
        # Sort by absolute correlation
        sorted_corr = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)
        
        print(f"Top {top_n} Features Correlated with Stock Price:\n")
        print(f"{'Feature':<30} {'Correlation':>12} {'Strength':>15}")
        print("-" * 60)
        
        for feature, corr in sorted_corr[:top_n]:
            direction = "positive" if corr > 0 else "negative"
            strength = "strong" if abs(corr) > 0.7 else "moderate" if abs(corr) > 0.4 else "weak"
            print(f"{feature:<30} {corr:>+12.3f} {strength:>10} {direction}")
        
        return dict(sorted_corr)
    
    def visualize_top_correlations(self, top_n=4):
        """Visualize top correlated features with stock price"""
        if self.merged_data is None:
            return
        
        print("\nGenerating correlation visualizations...")
        
        # Get top correlations
        numeric_cols = self.merged_data.select_dtypes(include=[np.number]).columns
        feature_cols = [col for col in numeric_cols if col != 'Stock_Price']
        
        correlations = {}
        for col in feature_cols:
            valid_data = self.merged_data[[col, 'Stock_Price']].dropna()
            if len(valid_data) > 1:
                correlations[col] = abs(valid_data[col].corr(valid_data['Stock_Price']))
        
        top_features = sorted(correlations.items(), key=lambda x: x[1], reverse=True)[:top_n]
        
        # Create subplots
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        fig.suptitle(f'{self.ticker} - Top Financial Features vs Stock Price', 
                     fontsize=16, fontweight='bold')
        
        axes = axes.flatten()
        
        for idx, (feature, _) in enumerate(top_features):
            if idx >= 4:
                break
            
            ax = axes[idx]
            valid_data = self.merged_data[[feature, 'Stock_Price', 'Date']].dropna()
            
            scatter = ax.scatter(valid_data[feature], valid_data['Stock_Price'],
                               s=100, alpha=0.6, c=range(len(valid_data)), cmap='viridis')
            
            ax.set_xlabel(feature, fontweight='bold')
            ax.set_ylabel('Stock Price ($)', fontweight='bold')
            ax.set_title(f'{feature} vs Stock Price', fontweight='bold')
            ax.grid(True, alpha=0.3)
            plt.colorbar(scatter, ax=ax, label='Time Period')
        
        plt.tight_layout()
        plt.show()
        print("✓ Visualizations generated")
    
    def generate_report(self):
        """Generate comprehensive analysis report"""
        print(f"\n{'='*60}")
        print(f"COMPREHENSIVE FINANCIAL ANALYSIS: {self.ticker}")
        print(f"{'='*60}\n")
        
        if self.merged_data is not None and len(self.merged_data) > 0:
            latest = self.merged_data.iloc[-1]
            
            print(" LATEST PERIOD METRICS:")
            print("-" * 60)
            print(f"Date: {latest['Date'].strftime('%Y-%m-%d')}")
            print(f"Stock Price: ${latest['Stock_Price']:.2f}\n")
            
            # Display key metrics
            key_metrics = [
                'Current_Ratio', 'Quick_Ratio', 'Debt_to_Equity',
                'ROE', 'ROA', 'Net_Profit_Margin', 'Operating_Margin'
            ]
            
            for metric in key_metrics:
                if metric in latest:
                    value = latest[metric]
                    if pd.notna(value):
                        if metric in ['ROE', 'ROA', 'Net_Profit_Margin', 'Operating_Margin']:
                            print(f"{metric}: {value:.2f}%")
                        else:
                            print(f"{metric}: {value:.2f}")
        
        print(f"\n{'='*60}")
        print("Analysis Complete!")
        print(f"{'='*60}")


In [18]:
# =============================================================================
# MAIN EXECUTION
# =============================================================================
import glob

def DumpDataforaticker(ticker):
    # Example usage
    #ticker = "MSFT"
    
    print(f"\n{'='*70}")
    print(f"MODULAR FINANCIAL ANALYSIS FOR {ticker}")
    print(f"{'='*70}\n")
    
    # Initialize analyzer
    analyzer = StockFinancialAnalyzer(ticker)
    
    # Fetch and prepare all data
    features = analyzer.fetch_and_prepare_data()
    
    # Show feature summary
    if analyzer.engineer:
        analyzer.engineer.get_feature_summary()
    
   # print(features.describe())
    print ( features.dtypes)
    # Merge with stock prices
    merged_data = analyzer.merge_with_stock_prices()
    
    # Analyze correlations
    correlations = analyzer.analyze_correlations(top_n=15)
    
    # Visualize top correlations
    #analyzer.visualize_top_correlations(top_n=4)

    # Generate comprehensive report
    analyzer.generate_report()
    
    # Display sample of merged data
    
    if merged_data is not None:
        print(f"\n{'='*60}")
        print("SAMPLE MERGED DATA")
        print(f"{'='*60}\n")
        print(merged_data.head())
        
        # Export to CSV
        output_dir = "financial_data"
        os.makedirs(output_dir, exist_ok=True)
        
        output_file = os.path.join(output_dir, f"{ticker}_financial_analysis.csv")
        merged_data.to_csv(output_file, index=False)
        
        print(f"\n✓ Data exported to: {output_file}")


# Read and combine all ticker CSV files
def load_all_tickers(pattern="*_financial_analysis.csv"):
    """Load all ticker financial data into a single dataframe"""
    data_dir = "financial_data/"
    full_pattern = os.path.join(data_dir, pattern)
    csv_files = glob.glob(full_pattern)
    
    if not csv_files:
        print("No CSV files found!")
        return None
    
    all_data = []
    for file in csv_files:
        print(file)
        fileName = file.split('\\')[1]
        ticker = fileName.split('_')[0]
        df = pd.read_csv(file)
        df['Ticker'] = ticker
        all_data.append(df)
        print(f"✓ Loaded {ticker}: {len(df)} records")
    
    combined = pd.concat(all_data, ignore_index=True)
    combined['Date'] = pd.to_datetime(combined['Date'])
    combined = combined.sort_values(['Ticker', 'Date']).reset_index(drop=True)
    
    print(f"\n✓ Combined dataset: {len(combined)} total records")
    print(f"✓ Tickers: {combined['Ticker'].unique().tolist()}")

    feature_cols = [ col for col in df.columns  if col not in ["Date", "Ticker", "Stock_Price"]]

    for col in feature_cols:
        if df[col].isna().sum() > 0:
            df[col + "_missing"] = df[col].isna().astype(int)

    output_file = f"combined_financial_analysis.csv"
    combined.to_csv(output_file, index=False)
    
    return combined
    

In [19]:
#load_all_tickers()

In [20]:
#popular_tickers = [ 'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA','META', 'TSLA', 'UNH', 'JPM']
#popular_tickers = [ 'NVDA','META', 'TSLA']
popular_tickers = df['Ticker'][0:100]

#popular_tickers = ['CCEP']

if __name__ == "__main__":
    for ticker in popular_tickers:
        try:
            DumpDataforaticker(ticker)
        except Exception as e:
            print(f"Skipping {ticker}: {e}")
    load_all_tickers()  
    
    


MODULAR FINANCIAL ANALYSIS FOR NVDA


Fetching all financial data for NVDA

Fetching quarterly balance sheet for NVDA...
✓ Retrieved 6 periods of balance sheet data
Fetching quarterly income statement for NVDA...
✓ Retrieved 5 periods of income statement data
Fetching quarterly cash flow for NVDA...
✓ Retrieved 5 periods of cash flow data
Fetching stock prices for NVDA (2y)...
 Retrieved 500 periods of price data
sector str Technology
Sector Info Technology

FEATURE ENGINEERING: Calculating All Financial Ratios

Calculating Liquidity Ratios for NVDA

  Calculating Current Ratio...
 Filled 1 missing values via interpolation
  Current Ratio calculated
  Range: 3.39 to 4.47

  Calculating Quick Ratio...
 Filled 1 missing values via interpolation
  Quick Ratio calculated
 Range: 2.96 to 3.88

  Calculating Cash Ratio...
 Filled 1 missing values via interpolation
 Cash Ratio calculated
  Range: 0.44 to 0.57

  Calculating Working Capital...
 Working Capital calculated

  Validating ratio c

C:\Users\radha\AppData\Local\Temp\ipykernel_18504\1229826883.py:166: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  series = series.fillna(method='ffill')
C:\Users\radha\AppData\Local\Temp\ipykernel_18504\1229826883.py:169: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  series = series.fillna(method='bfill')
C:\Users\radha\AppData\Local\Temp\ipykernel_18504\1229826883.py:166: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  series = series.fillna(method='ffill')
C:\Users\radha\AppData\Local\Temp\ipykernel_18504\1229826883.py:169: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  series = series.fillna(method='bfill')
C:\Users\radha\AppData\Local

✓ Retrieved 6 periods of balance sheet data
Fetching quarterly income statement for AAPL...
✓ Retrieved 5 periods of income statement data
Fetching quarterly cash flow for AAPL...
✓ Retrieved 7 periods of cash flow data
Fetching stock prices for AAPL (2y)...
 Retrieved 500 periods of price data
sector str Technology
Sector Info Technology

FEATURE ENGINEERING: Calculating All Financial Ratios

Calculating Liquidity Ratios for AAPL

  Calculating Current Ratio...
 Filled 1 missing values via interpolation
  Current Ratio calculated
  Range: 0.82 to 0.97

  Calculating Quick Ratio...
 Filled 1 missing values via interpolation
  Quick Ratio calculated
 Range: 0.78 to 0.94

  Calculating Cash Ratio...
 Filled 1 missing values via interpolation
 Cash Ratio calculated
  Range: 0.19 to 0.28

  Calculating Working Capital...
 Working Capital calculated

  Validating ratio constraints...
 All constraints satisfied

All liquidity ratios calculated

{'Current_Ratio': 2024-09-30    0.922938
2024-1

C:\Users\radha\AppData\Local\Temp\ipykernel_18504\1229826883.py:166: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  series = series.fillna(method='ffill')
C:\Users\radha\AppData\Local\Temp\ipykernel_18504\1229826883.py:169: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  series = series.fillna(method='bfill')
C:\Users\radha\AppData\Local\Temp\ipykernel_18504\1229826883.py:166: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  series = series.fillna(method='ffill')
C:\Users\radha\AppData\Local\Temp\ipykernel_18504\1229826883.py:169: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  series = series.fillna(method='bfill')
C:\Users\radha\AppData\Local

✓ Retrieved 7 periods of balance sheet data
Fetching quarterly income statement for MSFT...
✓ Retrieved 5 periods of income statement data
Fetching quarterly cash flow for MSFT...
✓ Retrieved 7 periods of cash flow data
Fetching stock prices for MSFT (2y)...
 Retrieved 500 periods of price data
sector str Technology
Sector Info Technology

FEATURE ENGINEERING: Calculating All Financial Ratios

Calculating Liquidity Ratios for MSFT

  Calculating Current Ratio...
 Filled 2 missing values via interpolation
  Current Ratio calculated
  Range: 1.35 to 1.40

  Calculating Quick Ratio...
 Filled 2 missing values via interpolation
  Quick Ratio calculated
 Range: 1.34 to 1.39

  Calculating Cash Ratio...
 Filled 2 missing values via interpolation
 Cash Ratio calculated
  Range: 0.16 to 0.25

  Calculating Working Capital...
 Working Capital calculated

  Validating ratio constraints...
 All constraints satisfied

All liquidity ratios calculated

{'Current_Ratio': 2024-06-30    1.350820
2024-0

...

get features             Current_Ratio  Quick_Ratio  Cash_Ratio  Working_Capital  \
2024-06-30       9.631257     9.631257    9.153875              NaN   
2024-09-30       9.631257     9.631257    9.153875     4.655700e+09   
2024-12-31       8.962882     8.962882    8.475466     4.397900e+09   
2025-03-31       6.009525     6.009525    5.081171     3.628900e+09   
2025-06-30       5.834456     5.834456    4.887662     3.589100e+09   

            Debt_to_Equity  Debt_to_Assets  Equity_Ratio       ROE       ROA  \
2024-06-30        0.219941        0.180288      0.819712  0.707506  0.579951   
2024-09-30        0.219941        0.180288      0.819712  0.707506  0.579951   
2024-12-31        0.225498        0.184005      0.815995  0.791686  0.646012   
2025-03-31        0.218777        0.179505      0.820495 -0.172974 -0.141924   
2025-06-30        0.221557        0.181373      0.818627  0.072081  0.059008   

            Net_Profit_Margin  ...  Revenue_Growth_QoQ  Net_Income_Growth_QoQ  

C:\Users\radha\AppData\Roaming\Python\Python39\site-packages\numpy\lib\function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\radha\AppData\Roaming\Python\Python39\site-packages\numpy\lib\function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
C:\Users\radha\AppData\Roaming\Python\Python39\site-packages\numpy\lib\function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\radha\AppData\Roaming\Python\Python39\site-packages\numpy\lib\function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
C:\Users\radha\AppData\Roaming\Python\Python39\site-packages\numpy\lib\function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\radha\AppData\Roaming\Python\Python39\site-packages\numpy\lib\function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= st

✓ Retrieved 7 periods of balance sheet data
Fetching quarterly income statement for ARM...
✓ Retrieved 5 periods of income statement data
Fetching quarterly cash flow for ARM...
✓ Retrieved 5 periods of cash flow data
Fetching stock prices for ARM (2y)...
 Retrieved 500 periods of price data
sector str Technology
Sector Info Technology

FEATURE ENGINEERING: Calculating All Financial Ratios

Calculating Liquidity Ratios for ARM

  Calculating Current Ratio...
 Filled 2 missing values via interpolation
  Current Ratio calculated
  Range: 4.96 to 5.59

  Calculating Quick Ratio...
 Filled 2 missing values via interpolation
  Quick Ratio calculated
 Range: 4.96 to 5.59

  Calculating Cash Ratio...
 Filled 2 missing values via interpolation
 Cash Ratio calculated
  Range: 1.89 to 2.66

  Calculating Working Capital...
 Working Capital calculated

  Validating ratio constraints...
 All constraints satisfied

All liquidity ratios calculated

{'Current_Ratio': 2024-06-30    4.958810
2024-09-30

C:\Users\radha\AppData\Local\Temp\ipykernel_18504\1229826883.py:166: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  series = series.fillna(method='ffill')
C:\Users\radha\AppData\Local\Temp\ipykernel_18504\1229826883.py:169: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  series = series.fillna(method='bfill')
C:\Users\radha\AppData\Local\Temp\ipykernel_18504\1229826883.py:166: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  series = series.fillna(method='ffill')
C:\Users\radha\AppData\Local\Temp\ipykernel_18504\1229826883.py:169: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  series = series.fillna(method='bfill')
C:\Users\radha\AppData\Local

✓ Loaded ADSK: 6 records
financial_data\AEP_financial_analysis.csv
✓ Loaded AEP: 7 records
financial_data\ALNY_financial_analysis.csv
✓ Loaded ALNY: 7 records
financial_data\AMAT_financial_analysis.csv
✓ Loaded AMAT: 7 records
financial_data\AMD_financial_analysis.csv
✓ Loaded AMD: 7 records
financial_data\AMGN_financial_analysis.csv
✓ Loaded AMGN: 6 records
financial_data\AMZN_financial_analysis.csv
✓ Loaded AMZN: 6 records
financial_data\APP_financial_analysis.csv
✓ Loaded APP: 7 records
financial_data\ARM_financial_analysis.csv
✓ Loaded ARM: 7 records
financial_data\ASML_financial_analysis.csv
✓ Loaded ASML: 7 records
financial_data\AVGO_financial_analysis.csv
✓ Loaded AVGO: 5 records
financial_data\AXON_financial_analysis.csv
✓ Loaded AXON: 6 records
financial_data\BKNG_financial_analysis.csv
✓ Loaded BKNG: 6 records
financial_data\BKR_financial_analysis.csv
✓ Loaded BKR: 7 records
financial_data\CDNS_financial_analysis.csv
✓ Loaded CDNS: 6 records
financial_data\CEG_financial_anal

In [ ]:

"""
url = "https://ftp.nasdaqtrader.com/dynamic/SymDir/nasdaqlisted.txt"
df = pd.read_csv(url, sep="|")

# Remove last footer row
df = df[df["Symbol"] != "File Creation Time"]

nasdaq_tickers = df["Symbol"].tolist()

print(len(nasdaq_tickers))
print(nasdaq_tickers[:10])
"""


In [ ]:

def get_market_caps(tickers, batch_size=50):
    results = []

    # ensure all tickers are strings
    tickers = [str(t) for t in tickers if isinstance(t, str)]

    for i in range(0, len(tickers), batch_size):
        batch = tickers[i:i + batch_size]

        data = yf.Tickers(" ".join(batch))

        for symbol in batch:
            try:
                info = data.tickers[symbol].fast_info
                market_cap = info.get("market_cap")
                if market_cap:
                    results.append({
                        "Ticker": symbol,
                        "MarketCap": market_cap
                    })
            except Exception:
                pass

    return pd.DataFrame(results)

    

In [ ]:
import yfinance as yf
import time

data = []

for symbol in tickers:
    try:
        t = yf.Ticker(symbol)
        mc = t.fast_info.get("market_cap")

        if mc is None:
            mc = t.info.get("marketCap")  # fallback
            
        info = t.info
        total_debt = info.get("totalDebt", 0)
        cash = info.get("totalCash", 0)

        if market_cap and total_debt is not None:
            net_debt = max(total_debt - cash, 0)

            data.append({"Ticker": symbol,
                "MarketCap": market_cap,
                "NetDebt": net_debt,
                "NetDebt_to_MarketCap": net_debt / market_cap
            })

        time.sleep(0.3)  # VERY IMPORTANT

    except Exception:
        continue

In [ ]:
mc_df = pd.DataFrame(data)

print(len(mc_df), mc_df.columns)

In [ ]:

mc_df.head()


In [ ]:
#top_500 = mc_df.sort_values("MarketCap", ascending=False).head(500).reset_index(drop=True)
#print(top_500.head(10))

df_sorted = ( df.sort_values( by=["MarketCap", "NetDebt_to_MarketCap"], ascending=[False, True]))

print(df_sorted.head(10))


In [ ]:
# Normalize values
df["MC_rank"] = df["MarketCap"].rank(ascending=False)
df["Debt_rank"] = df["NetDebt_to_MarketCap"].rank(ascending=True)

# Combined score (lower is better)
df["Score"] = df["MC_rank"] + df["Debt_rank"]

df_sorted = df.sort_values("Score")


In [ ]:
mc_df = pd.DataFrame(data)

print(len(mc_df), mc_df.columns)



In [ ]:
mc_df = get_market_caps(tickers)
print(mc_df.head())
print(mc_df.columns)
print(len(mc_df))


In [ ]:

top_500 = mc_df.sort_values("MarketCap", ascending=False).head(500)
print(top_500.head(10))


In [ ]:
popular_tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA','META', 'TSLA', 'TLN' ]

if __name__ == "__main__":
    for ticker in df['Ticker'][0:100]:
        DumpDataforaticker(ticker)
    df = load_all_tickers()
    print(df.Ticker.unique())
    

In [ ]:
import pandas as pd

url = "https://www.nasdaqtrader.com/dynamic/SymDir/nasdaqlisted.txt"

df = pd.read_csv(url, sep="|")
df = df[df["Symbol"] != "File Creation Time"]

tickers = df["Symbol"].tolist()
print(tickers[:500])





In [ ]:
# ============================================================================
# EXAMPLE USAGE
# ============================================================================
'''
if __name__ == "__main__":
    
    # Simulate balance sheet with edge cases
    dates = pd.date_range('2023-01-01', periods=6, freq='Q')
    
    balance_sheet = pd.DataFrame({
        dates[0]: {
            'Total Assets': 200e6,
            'Total Liabilities Net Minority Interest': 80e6,
            'Stockholders Equity': 120e6  # Normal
        },
        dates[1]: {
            'Total Assets': 210e6,
            'Total Liabilities Net Minority Interest': 210e6,
            'Stockholders Equity': 0  # ZERO EQUITY!
        },
        dates[2]: {
            'Total Assets': 220e6,
            'Total Liabilities Net Minority Interest': 230e6,
            'Stockholders Equity': -10e6  # NEGATIVE EQUITY (insolvent)
        },
        dates[3]: {
            'Total Assets': np.nan,
            'Total Liabilities Net Minority Interest': np.nan,
            'Stockholders Equity': np.nan  # MISSING DATA
        },
        dates[4]: {
            'Total Assets': 230e6,
            'Total Liabilities Net Minority Interest': 100e6,
            'Stockholders Equity': 130e6  # Recovered
        },
        dates[5]: {
            'Total Assets': 240e6,
            'Total Liabilities Net Minority Interest': 110e6,
            'Stockholders Equity': 130e6  # Normal
        }
    }).T
    
    print("Balance Sheet Data:")
    print(balance_sheet)
    
    # Calculate leverage ratios
    calculator = RobustLeverageCalculator('TEST', balance_sheet)
    ratios = calculator.calculate_all_leverage_ratios()
    
    # Display results
    results_df = pd.DataFrame(ratios)
    print("\nCalculated Ratios:")
    print(results_df)
    
    # Data quality report
    report = calculator.get_data_quality_report()
    print("\n" + "="*60)
    print("DATA QUALITY REPORT")
    print("="*60)
    print(f"Total Periods: {report['total_periods']}")
    print(f"Issues Found: {len(report['issues'])}")
    for issue in report['issues']:
        print(f"\n• {issue['issue']}")
        print(f"  Count: {issue['count']} periods")
        print(f"  Severity: {issue['severity']}")
        print(f"  Action: {issue['action']}")
'''

In [ ]:
'''

    def calculate_profitability_ratios(self):
        """Calculate profitability ratios (requires income statement and balance sheet)"""
        if self.income_statement is None or self.balance_sheet is None:
            print("     Income statement and balance sheet required for profitability ratios")
            return None
        
        print("\nCalculating profitability ratios...")
        ratios = {}
        
        try:
            # Get common dates
            common_dates = self.balance_sheet.columns.intersection(self.income_statement.columns)
            
            if len(common_dates) == 0:
                print(" No overlapping dates between balance sheet and income statement")
                return None
            
            # ROA = Net Income / Total Assets
            if 'Net Income' in self.income_statement.index and 'Total Assets' in self.balance_sheet.index:
                net_income = self.income_statement.loc['Net Income', common_dates]
                total_assets = self.balance_sheet.loc['Total Assets', common_dates]
                ratios['ROA'] = (net_income / total_assets) * 100
                print("  ✓ ROA (Return on Assets)")
            
            # ROE = Net Income / Stockholders Equity
            if 'Net Income' in self.income_statement.index and 'Stockholders Equity' in self.balance_sheet.index:
                net_income = self.income_statement.loc['Net Income', common_dates]
                equity = self.balance_sheet.loc['Stockholders Equity', common_dates]
                ratios['ROE'] = (net_income / equity) * 100
                print("  ✓ ROE (Return on Equity)")
            
            # Asset Turnover = Revenue / Total Assets
            if 'Total Revenue' in self.income_statement.index and 'Total Assets' in self.balance_sheet.index:
                revenue = self.income_statement.loc['Total Revenue', common_dates]
                total_assets = self.balance_sheet.loc['Total Assets', common_dates]
                ratios['Asset_Turnover'] = revenue / total_assets
                print("  ✓ Asset Turnover")
            
            # Net Profit Margin = Net Income / Revenue
            if 'Net Income' in self.income_statement.index and 'Total Revenue' in self.income_statement.index:
                net_income = self.income_statement.loc['Net Income', common_dates]
                revenue = self.income_statement.loc['Total Revenue', common_dates]
                ratios['Net_Profit_Margin'] = (net_income / revenue) * 100
                print("  ✓ Net Profit Margin")
            
            # Gross Profit Margin
            if 'Gross Profit' in self.income_statement.index and 'Total Revenue' in self.income_statement.index:
                gross_profit = self.income_statement.loc['Gross Profit', common_dates]
                revenue = self.income_statement.loc['Total Revenue', common_dates]
                ratios['Gross_Profit_Margin'] = (gross_profit / revenue) * 100
                print("  ✓ Gross Profit Margin")
            
            # Operating Margin
            if 'Operating Income' in self.income_statement.index and 'Total Revenue' in self.income_statement.index:
                operating_income = self.income_statement.loc['Operating Income', common_dates]
                revenue = self.income_statement.loc['Total Revenue', common_dates]
                ratios['Operating_Margin'] = (operating_income / revenue) * 100
                print("  ✓ Operating Margin")
            
            # EPS (Earnings Per Share)
            if 'Diluted EPS' in self.income_statement.index:
                ratios['EPS'] = self.income_statement.loc['Diluted EPS', common_dates]
                print("  ✓ EPS (Earnings Per Share)")
            
            return pd.DataFrame(ratios)
            
        except Exception as e:
            print(f"✗ Error calculating profitability ratios: {e}")
            import traceback
            traceback.print_exc()
            return None

'''

In [ ]:
# ============================================================================
# EXAMPLE USAGE
# ============================================================================
'''
def example_usage():
    """
    Example with problematic data
    """
    
    # Simulate balance sheet with zero Current Liabilities
    dates = pd.date_range('2023-01-01', periods=4, freq='Q')
    
    balance_sheet = pd.DataFrame({
        dates[0]: {
            'Current Assets': 100_000_000,
            'Current Liabilities': 50_000_000,  # Normal
            'Inventory': 20_000_000,
            'Cash And Cash Equivalents': 30_000_000
        },
        dates[1]: {
            'Current Assets': 110_000_000,
            'Current Liabilities': 0,  # ZERO! Problem case
            'Inventory': 22_000_000,
            'Cash And Cash Equivalents': 35_000_000
        },
        dates[2]: {
            'Current Assets': 120_000_000,
            'Current Liabilities': np.nan,  # MISSING! Problem case
            'Inventory': 25_000_000,
            'Cash And Cash Equivalents': 40_000_000
        },
        dates[3]: {
            'Current Assets': 130_000_000,
            'Current Liabilities': 60_000_000,  # Normal
            'Inventory': 28_000_000,
            'Cash And Cash Equivalents': 45_000_000
        }
    }).T
    
    print("Balance Sheet Data:")
    print(balance_sheet)
    print("\n")
    
    # Calculate ratios
    calculator = RobustLiquidityCalculator('TEST', balance_sheet)
    ratios = calculator.calculate_all_liquidity_ratios()
    
    # Display results
    results_df = pd.DataFrame(ratios)
    print("\nCalculated Ratios:")
    print(results_df)
    
    # Data quality report
    report = calculator.get_data_quality_report()
    print("\nData Quality Report:")
    print(f"Total Periods: {report['total_periods']}")
    print(f"Issues Found: {len(report['issues'])}")
    for issue in report['issues']:
        print(f"  • {issue['issue']}: {issue['count']} periods ({issue['severity']})")
        print(f"    → {issue['action']}")

'''

In [ ]:
'''
# ============================================================================
# EXAMPLE USAGE
# ============================================================================

if __name__ == "__main__":
    
    # Create sample data with various edge cases
    dates = pd.date_range('2023-01-01', periods=8, freq='Q')
    
    income_statement = pd.DataFrame({
        dates[0]: {'Total Revenue': 100e6, 'Net Income': 15e6, 'Operating Income': 25e6, 'Basic EPS': 1.50},
        dates[1]: {'Total Revenue': 0, 'Net Income': -5e6, 'Operating Income': -3e6, 'Basic EPS': -0.50},  # ZERO REVENUE
        dates[2]: {'Total Revenue': 110e6, 'Net Income': 18e6, 'Operating Income': 28e6, 'Basic EPS': 1.80},  # RECOVERY
        dates[3]: {'Total Revenue': 115e6, 'Net Income': -2e6, 'Operating Income': 5e6, 'Basic EPS': -0.20},  # LOSS
        dates[4]: {'Total Revenue': 120e6, 'Net Income': 20e6, 'Operating Income': 30e6, 'Basic EPS': 2.00},
        dates[5]: {'Total Revenue': 125e6, 'Net Income': 22e6, 'Operating Income': 32e6, 'Basic EPS': 2.20},
        dates[6]: {'Total Revenue': 130e6, 'Net Income': 24e6, 'Operating Income': 34e6, 'Basic EPS': 2.40},
        dates[7]: {'Total Revenue': 135e6, 'Net Income': 26e6, 'Operating Income': 36e6, 'Basic EPS': 2.60},
    }).T
    
    balance_sheet = pd.DataFrame({
        dates[0]: {'Total Assets': 200e6, 'Stockholders Equity': 120e6},
        dates[1]: {'Total Assets': 195e6, 'Stockholders Equity': 115e6},  # DECLINE
        dates[2]: {'Total Assets': 210e6, 'Stockholders Equity': 125e6},
        dates[3]: {'Total Assets': 215e6, 'Stockholders Equity': 123e6},
        dates[4]: {'Total Assets': 220e6, 'Stockholders Equity': 130e6},
        dates[5]: {'Total Assets': 225e6, 'Stockholders Equity': 135e6},
        dates[6]: {'Total Assets': 230e6, 'Stockholders Equity': 140e6},
        dates[7]: {'Total Assets': 235e6, 'Stockholders Equity': 145e6},
    }).T
    
    print("="*60)
    print("TESTING ROBUST GROWTH CALCULATOR")
    print("="*60)
    
    # Create calculator
    growth_calc = RobustGrowthCalculator('TEST', income_statement, balance_sheet)
    
    # Calculate all growth metrics
    growth_metrics = growth_calc.calculate_all_ratios()
    
    # Display results
    print("\nGrowth Metrics (%):")
    growth_df = pd.DataFrame(growth_metrics).round(1)
    print(growth_df)
    
    # Data quality report
    report = growth_calc.get_data_quality_report()
    print("\n" + "="*60)
    print("DATA QUALITY REPORT")
    print("="*60)
    print(f"Total Periods: {report['total_periods']}")
    print(f"Issues Found: {len(report['issues'])}")
    
    for issue in report['issues']:
        print(f"\n• {issue['issue']}")
        print(f"  Count: {issue['count']}")
        print(f"  Severity: {issue['severity']}")
        print(f"  Action: {issue['action']}")
'''

In [ ]:
'''
def calculate_efficiency_ratios(self):
        """Calculate efficiency ratios"""
        if self.income_statement is None or self.balance_sheet is None:
            print(" Income statement and balance sheet required for efficiency ratios")
            return None
        
        print("\nCalculating efficiency ratios...")
        ratios = {}
        
        try:
            common_dates = self.balance_sheet.columns.intersection(self.income_statement.columns)
            
            # Inventory Turnover = Cost of Revenue / Inventory
            if 'Cost Of Revenue' in self.income_statement.index and 'Inventory' in self.balance_sheet.index:
                cogs = self.income_statement.loc['Cost Of Revenue', common_dates]
                inventory = self.balance_sheet.loc['Inventory', common_dates]
                ratios['Inventory_Turnover'] = cogs / inventory
                print("  ✓ Inventory Turnover")
            
            # Receivables Turnover = Revenue / Accounts Receivable
            if 'Total Revenue' in self.income_statement.index and 'Accounts Receivable' in self.balance_sheet.index:
                revenue = self.income_statement.loc['Total Revenue', common_dates]
                receivables = self.balance_sheet.loc['Accounts Receivable', common_dates]
                ratios['Receivables_Turnover'] = revenue / receivables
                print("  ✓ Receivables Turnover")
            
            return pd.DataFrame(ratios)
            
        except Exception as e:
            print(f"✗ Error calculating efficiency ratios: {e}")
            return None
    
    def calculate_growth_metrics(self):
        """Calculate growth metrics (quarter-over-quarter and year-over-year)"""
        print("\nCalculating growth metrics...")
        growth_metrics = {}
        
        try:
            # Revenue Growth
            if self.income_statement is not None and 'Total Revenue' in self.income_statement.index:
                revenue = self.income_statement.loc['Total Revenue']
                growth_metrics['Revenue_Growth_QoQ'] = revenue.pct_change(periods=-1) * 100
                print("  ✓ Revenue Growth (QoQ)")
            
            # Net Income Growth
            if self.income_statement is not None and 'Net Income' in self.income_statement.index:
                net_income = self.income_statement.loc['Net Income']
                growth_metrics['Net_Income_Growth_QoQ'] = net_income.pct_change(periods=-1) * 100
                print("  ✓ Net Income Growth (QoQ)")
            
            # Total Assets Growth
            if self.balance_sheet is not None and 'Total Assets' in self.balance_sheet.index:
                assets = self.balance_sheet.loc['Total Assets']
                growth_metrics['Assets_Growth_QoQ'] = assets.pct_change(periods=-1) * 100
                print("  ✓ Assets Growth (QoQ)")             
            
        except Exception as e:
            print(f"✗ Error calculating growth metrics: {e}")
            return None
'''


In [ ]:
"""
                            'industry_key': info.get('industryKey', 'Unknown'),
                            'sector_key': info.get('sectorKey', 'Unknown'),
                            'country': info.get('country', 'Unknown'),
                            'exchange': info.get('exchange', 'Unknown'),
                            'company_name': info.get('longName', info.get('shortName', ticker)),
                            'market_cap': info.get('marketCap', None),
                            'employees': info.get('fullTimeEmployees', None),
                            'business_summary': info.get('longBusinessSummary', ''),
"""
                       
            
"""
                'industry_key': 'Unknown',
                'sector_key': 'Unknown',
                'country': 'Unknown',
                'exchange': 'Unknown',
                'company_name': ticker,
                'market_cap': None,
                'employees': None,
                'business_summary': '',
"""